# Nigeria–UK Cancer Diagnostic Pathways — Reproducible Analysis

**Research analysis documented:** 9 November 2026

This notebook documents a comparative scoping-review analysis of cancer diagnostic pathways in Nigeria and the United Kingdom. The analytical workflow included source-data verification, descriptive study mapping, assessment of diagnostic-pathway reporting, Nigerian regional comparisons, thematic coding, and preliminary examination of intervention evidence.

Four cleaned extraction datasets were analysed without overwriting their source files. A separate thematic coding workbook provided the classification framework. The final analytical dataset comprised **242 unique publications** (42 Nigerian and 200 UK publications), with **979 factor-level entries** and **472 problem/intervention-level entries**.

An earlier pilot involving 20 findings informed the thematic coding workflow. Subsequently, **68 coding decisions** were reviewed and approved on 9 November 2026: 61 findings received thematic codes and seven remained unclassified. The approved decisions and their rationales were preserved in the analysis.

The notebook also contains subsequent analysis cells for RQ1–RQ4. Its RQ4 output identifies **91 entries requiring further evidence interpretation**; that screening output is not itself a completed effectiveness assessment of all 472 entries.


## 0. Input verification and file manifest

The five named input files were checked for presence before analysis. File sizes and SHA-256 hashes were recorded to identify the exact study datasets and thematic coding framework used. The framework workbook supplied coding rules and was not counted as an additional evidence dataset.


In [ ]:
import os, re
from pathlib import Path
from collections import Counter
import pandas as pd
from IPython.display import display

print('Python/pandas:', __import__('sys').version.split()[0], pd.__version__)
print('Working directory:', os.getcwd())
print('Excel files present:', [p.name for p in Path('/content').glob('*.xlsx')])

Python/pandas: 3.13.18 2.2.3
Working directory: /content
Excel files present: ['Sheet_4_clean(2).xlsx', 'Nigeria_UK_Thematic_Coding_Framework.xlsx', 'Sheet_1_clean(2).xlsx', 'Sheet_2_clean(3).xlsx', 'Sheet_3_clean(2).xlsx']


In [ ]:
from pathlib import Path
import hashlib

EXPECTED = {
    'sheet1': 'Sheet_1_clean(2).xlsx',
    'sheet2': 'Sheet_2_clean(3).xlsx',
    'sheet3': 'Sheet_3_clean(2).xlsx',
    'sheet4': 'Sheet_4_clean(2).xlsx',
    'codebook': 'Nigeria_UK_Thematic_Coding_Framework.xlsx',
}
ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
PATHS = {k: ROOT / fname for k, fname in EXPECTED.items()}
missing = [str(p.name) for p in PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError('Upload these REQUIRED versions to Colab: ' + ', '.join(missing))

manifest = pd.DataFrame([
    {'dataset': key, 'exact_filename': path.name,
     'bytes': path.stat().st_size,
     'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    for key,path in PATHS.items()
])
display(manifest)
s1 = pd.read_excel(PATHS['sheet1'])
s2 = pd.read_excel(PATHS['sheet2'])
s3 = pd.read_excel(PATHS['sheet3'])
s4 = pd.read_excel(PATHS['sheet4'])
print('Original worksheet data rows:', {f'Sheet {i}':len(x) for i,x in enumerate([s1,s2,s3,s4],1)})

,dataset,exact_filename,bytes,sha256
0,sheet1,Sheet_1_clean(2).xlsx,83899,31833620048e3048984a8f47f13761f629c476b15d0a5d...
1,sheet2,Sheet_2_clean(3).xlsx,102250,43db4056f772b05ae6616e2e8d90a2f0cb348156c4bf31...
2,sheet3,Sheet_3_clean(2).xlsx,129810,ec50b122da188219567fb18ab7c0d27591fd2195ad67b6...
3,sheet4,Sheet_4_clean(2).xlsx,87818,3a9369f8adb7054eb79bab8690c7e06522b92225fcb388...
4,codebook,Nigeria_UK_Thematic_Coding_Framework.xlsx,10697,31a21d63b80a60c3ef97240ee1c8f50f80bdc904ba7067...


Original worksheet data rows: {'Sheet 1': 242, 'Sheet 2': 242, 'Sheet 3': 979, 'Sheet 4': 472}


## 1. Study identifiers and dataset integrity

Study IDs were standardized and checked across all four datasets. The study-level evidence comprised **42 Nigerian and 200 UK publications**. The finding-level datasets contained **979 factors** in Sheet 3 and **472 problem/intervention entries** in Sheet 4. Cross-sheet checks identified missing or unmatched study identifiers and distinguished study counts from extracted-row counts.


In [ ]:
def normalized_id(v):
    if pd.isna(v): return None
    v = str(v).strip().upper().replace(' ', '')
    return v if re.fullmatch(r'(NG\d{2,3}|UK\d{3})', v) else None

id_columns={'sheet1':'study_id','sheet2':'study_id_clean',
            'sheet3':'study_id_clean','sheet4':'Study ID'}
data={'sheet1':s1,'sheet2':s2,'sheet3':s3,'sheet4':s4}
for key,df in data.items():
    col = id_columns[key]
    assert col in df.columns, f'{key}: study ID column missing: {col}'
    df['analysis_study_id'] = df[col].map(normalized_id)
    excluded = len(df)-df['analysis_study_id'].notna().sum()
    if excluded:
        raise ValueError(f'{key}: {excluded} invalid or blank IDs; STOP and investigate rather than silently exclude')
    df['analysis_country'] = df['analysis_study_id'].str[:2].map({'NG':'Nigeria','UK':'UK'})
    print(f'{key}: {len(df)} eligible rows, {df.analysis_study_id.nunique()} studies; '
          f'{excluded} blank/nonmatching rows excluded')

base_ids=set(s1.analysis_study_id)
print('Sheet 1 study counts:', s1.groupby('analysis_country')['analysis_study_id'].nunique().to_dict())
print('All Sheet 1 unique studies:',len(base_ids))
for key in ('sheet2','sheet3','sheet4'):
    unknown = set(data[key].analysis_study_id)-base_ids
    missing_here = base_ids-set(data[key].analysis_study_id)
    print(f'{key} IDs not in Sheet 1:', sorted(unknown), '| IDs absent:', sorted(missing_here))
    assert not unknown and not missing_here, f'{key}: study IDs differ from authoritative Sheet 1'
    if unknown:
        print('  ⚠ Inspect the corresponding original Study ID and the current cleaned ID; do not silently rename records.')

assert not s1.analysis_study_id.duplicated().any(), 'Sheet 1 has duplicate study IDs; inspect before analysis'
# Record mismatch rather than silently fixing the data.
assert len(base_ids)==242, 'Expected 242 studies; verify study IDs before proceeding'
assert s1.groupby('analysis_country').analysis_study_id.nunique().to_dict()=={'Nigeria':42,'UK':200}, 'Country counts mismatch'
assert (len(s1),len(s2),len(s3),len(s4))==(242,242,979,472), 'Row counts changed: inspect files'
print('Authoritative file ID and record checks PASSED')

sheet1: 242 eligible rows, 242 studies; 0 blank/nonmatching rows excluded
sheet2: 242 eligible rows, 242 studies; 0 blank/nonmatching rows excluded
sheet3: 979 eligible rows, 242 studies; 0 blank/nonmatching rows excluded
sheet4: 472 eligible rows, 242 studies; 0 blank/nonmatching rows excluded
Sheet 1 study counts: {'Nigeria': 42, 'UK': 200}
All Sheet 1 unique studies: 242
sheet2 IDs not in Sheet 1: [] | IDs absent: []
sheet3 IDs not in Sheet 1: [] | IDs absent: []
sheet4 IDs not in Sheet 1: [] | IDs absent: []
Authoritative file ID and record checks PASSED


## 2. Descriptive mapping of the included studies

Study-level frequencies were calculated by country and selected study characteristics. Each unique publication was counted once within the relevant descriptive summary. Extracted geographical categories, including multiregional categories, were preserved for subsequent review.


In [ ]:
def frequency(df,column,label):
    if column not in df.columns:
        print(f'SKIPPED {label}: column {column!r} not present')
        return pd.DataFrame()
    x=(df[['analysis_country','analysis_study_id',column]]
       .drop_duplicates(['analysis_country','analysis_study_id']))
    x[column]=x[column].fillna('NR').astype(str).str.strip().replace({'':'NR'})
    out=(x.groupby(['analysis_country',column])['analysis_study_id']
         .nunique().reset_index(name='unique_studies')
         .sort_values(['analysis_country','unique_studies'],ascending=[True,False]))
    print('\n'+label);display(out.head(30));return out

DESCRIPTIVE = {
    'Publication year':frequency(s1,'publication_year_clean','Publication year'),
    'Cancer type':frequency(s1,'cancer_type_clean','Cancer type'),
    'Study design':frequency(s1,'study_design_12cat_clean','Study design'),
    'Healthcare setting':frequency(s1,'healthcare_setting_clean','Healthcare setting'),
    'Data source':frequency(s1,'data_source_clean','Data source'),
    'Nigeria region':frequency(s1[s1.analysis_country.eq('Nigeria')], 'nigeria_region_clean','Nigeria region'),
    'UK nation':frequency(s1[s1.analysis_country.eq('UK')], 'uk_nation_clean','UK nation')
}
print('\nDenominator (distinct included studies):',len(base_ids))


Publication year


,analysis_country,publication_year_clean,unique_studies
11,Nigeria,2021,6
14,Nigeria,2024,6
5,Nigeria,2015,4
15,Nigeria,2025,4
0,Nigeria,2010,3
3,Nigeria,2013,3
7,Nigeria,2017,3
13,Nigeria,2023,3
1,Nigeria,2011,2
6,Nigeria,2016,2



Cancer type


,analysis_country,cancer_type_clean,unique_studies
1,Nigeria,breast,17
6,Nigeria,multiple_cancers,10
9,Nigeria,prostate,5
2,Nigeria,cervical,2
8,Nigeria,ovarian,2
0,Nigeria,all_cancers,1
3,Nigeria,gastrointestinal,1
4,Nigeria,laryngeal_cancer,1
5,Nigeria,lung,1
7,Nigeria,oral,1



Study design


,analysis_country,study_design_12cat_clean,unique_studies
7,Nigeria,secondary_data_analysis,12
3,Nigeria,cross_sectional,9
5,Nigeria,other_empirical,9
4,Nigeria,mixed_methods_study,5
6,Nigeria,qualitative_study,4
0,Nigeria,case_control,1
1,Nigeria,clinical_audit,1
2,Nigeria,cohort,1
10,UK,cohort,32
11,UK,cross_sectional,28


... (5 outputs left out)

,analysis_country,nigeria_region_clean,unique_studies
9,Nigeria,south_west,20
7,Nigeria,south_east,6
8,Nigeria,south_south,4
5,Nigeria,north_west,3
1,Nigeria,national_or_multiregional,2
2,Nigeria,north_central,2
6,Nigeria,north_west;south_west,2
0,Nigeria,NR,1
3,Nigeria,north_central;north_west;south_west,1
4,Nigeria,north_central;south_west,1



UK nation


,analysis_country,uk_nation_clean,unique_studies
0,UK,england,131
4,UK,uk_unspecified,50
3,UK,scotland,12
1,UK,multiple_uk_nations,3
2,UK,northern ireland,3
5,UK,wales,1



Denominator (distinct included studies): 242


## 3. Diagnostic-pathway reporting assessment (RQ1)

The standardized Sheet 2 categories were examined for eight diagnostic-pathway components. Reporting was classified as **explicitly described**, **broad or unclear**, or **not reported/not applicable**.

These classifications measured the specificity of information available in included publications, not the proportion of patients who followed a particular pathway. An explicitly classifiable category indicated that a sufficiently specific description was recorded; it did not establish that the corresponding event occurred for every patient.


In [ ]:
from collections import Counter

AUDIT_COLS = {
 'Symptom recognition':'symptom_recognition_clean',
 'Appraisal':'appraisal_clean',
 'Initial action':'initial_action_clean',
 'First healthcare contact':'first_healthcare_contact_clean',
 'Referral initiated':'referral_initiated_clean',
 'Diagnostic investigation':'diagnostic_investigation_clean',
 'Diagnosis confirmation':'diagnosis_confirmation_clean',
 'Route to diagnosis':'route_to_diagnosis_clean',
}
for stage, col in AUDIT_COLS.items():
    print('\n', stage)
    display(s2.groupby(['analysis_country',col],dropna=False).size().reset_index(name='studies').sort_values(['analysis_country','studies'],ascending=[True,False]))



 Symptom recognition


,analysis_country,symptom_recognition_clean,studies
3,Nigeria,symptom_or_change_described,29
2,Nigeria,not_reported,10
1,Nigeria,mixed_symptomatic_and_nonsymptomatic,2
0,Nigeria,descriptive_not_classifiable,1
9,UK,symptom_or_change_described,124
7,UK,not_applicable,35
4,UK,descriptive_not_classifiable,30
8,UK,not_reported,6
5,UK,mixed_symptomatic_and_nonsymptomatic,4
6,UK,nonsymptomatic_or_screen_detected,1



 Appraisal


,analysis_country,appraisal_clean,studies
5,Nigeria,not_reported,22
0,Nigeria,appraisal_described_other_or_unclear,10
4,Nigeria,multiple_appraisal_themes,5
6,Nigeria,spiritual_or_supernatural_attribution,2
1,Nigeria,fear_or_fatalism,1
2,Nigeria,limited_awareness_or_uncertainty,1
3,Nigeria,low_perceived_seriousness,1
7,UK,appraisal_described_other_or_unclear,169
10,UK,not_applicable,28
8,UK,fear_or_fatalism,1



 Initial action


,analysis_country,initial_action_clean,studies
6,Nigeria,not_reported,17
5,Nigeria,multiple_formal_and_informal_actions,6
0,Nigeria,action_described_other_or_unclear,5
2,Nigeria,formal_healthcare_contact_or_referral,5
4,Nigeria,informal_or_alternative_care,4
3,Nigeria,help_seeking_unspecified,3
1,Nigeria,clinician_referral_or_pathway_event,2
8,UK,clinician_referral_or_pathway_event,61
9,UK,formal_healthcare_contact_or_referral,53
7,UK,action_described_other_or_unclear,37


... (6 outputs left out)


 Diagnosis confirmation


,analysis_country,diagnosis_confirmation_clean,studies
6,Nigeria,pathology_or_cytology_reported,26
2,Nigeria,not_reported,5
1,Nigeria,narrative_or_unclear,4
4,Nigeria,pathology_and_clinical_reported,4
0,Nigeria,clinical_confirmation_reported,1
3,Nigeria,not_reported_with_context,1
5,Nigeria,pathology_and_imaging_reported,1
9,UK,narrative_or_unclear,135
10,UK,not_applicable,26
11,UK,not_reported,25



 Route to diagnosis


,analysis_country,route_to_diagnosis_clean,studies
1,Nigeria,narrative_or_unclear,19
3,Nigeria,other_clinical_route_mentioned,17
0,Nigeria,blank,2
2,Nigeria,not_reported,2
4,Nigeria,primary_care_route_mentioned,1
5,Nigeria,screening_route_mentioned,1
8,UK,narrative_or_unclear,77
12,UK,urgent_suspected_cancer_route,41
6,UK,blank,27
7,UK,emergency_route_mentioned,22


In [ ]:
# These are COMPLETE category rules for the eight RQ1 pathway components.
# Broad/uncertain labels are not counted as explicit evidence.
EXPLICIT = {
 'symptom_recognition_clean': {'symptom_or_change_described','mixed_symptomatic_and_nonsymptomatic','nonsymptomatic_or_screen_detected'},
 'appraisal_clean': {'multiple_appraisal_themes','spiritual_or_supernatural_attribution','low_perceived_seriousness','limited_awareness_or_uncertainty','fear_or_fatalism'},
 'initial_action_clean': {'multiple_formal_and_informal_actions','formal_healthcare_contact_or_referral','informal_or_alternative_care','help_seeking_unspecified','clinician_referral_or_pathway_event'},
 'first_healthcare_contact_clean': {'hospital_or_formal_care_level_unspecified','specialist_or_tertiary_care','multiple_or_variable_first_contacts','alternative_or_traditional_care','emergency_department','primary_care_or_gp','community_pharmacy_or_chemist','community_or_nonfacility'},
 'referral_initiated_clean': {'referral_described'},
 'diagnostic_investigation_clean': {'imaging','pathology_or_biopsy','laboratory_or_biomarker','endoscopy','clinical_examination'},
 'diagnosis_confirmation_clean': {'pathology_or_cytology_reported','pathology_and_clinical_reported','pathology_and_imaging_reported','clinical_confirmation_reported'},
 'route_to_diagnosis_clean': {'other_clinical_route_mentioned','primary_care_route_mentioned','screening_route_mentioned','urgent_suspected_cancer_route','emergency_route_mentioned'},
}
# The following are descriptions that cannot support an explicit stage/route claim on their own.
UNCLEAR_TOKENS = {'other_or_unclear','narrative_or_unclear','narrative_requires_review',
 'descriptive_not_classifiable','appraisal_described_other_or_unclear','action_described_other_or_unclear',
 'not_determined_from_description','described_incompletely','imaging_reported_confirmation_unclear'}

MISSING_TOKENS = {'','nr','na','n/a','none','nan','blank','not_reported','not_applicable',
 'not_reported_or_not_measured','not_reported_with_context'}

def clear_status(col, value):
    if pd.isna(value): return 'not reported / NA'
    terms=[z.strip().casefold() for z in str(value).split(';') if z.strip()]
    if not terms: return 'not reported / NA'
    if all(z in MISSING_TOKENS or z.startswith(('not_reported','not_applicable','not_measured')) for z in terms):
        return 'not reported / NA'
    if all(z in EXPLICIT[col] for z in terms): return 'explicitly described'
    return 'broad / unclear'

coverage_audited=[]
for stage,col in AUDIT_COLS.items():
    for country,g in s2.groupby('analysis_country'):
        statuses=g[col].map(lambda v:clear_status(col,v))
        c=statuses.value_counts()
        coverage_audited.append(dict(country=country,stage=stage,
            explicit=int(c.get('explicitly described',0)),
            broad_unclear=int(c.get('broad / unclear',0)),
            not_reported_NA=int(c.get('not reported / NA',0)),
            denominator=len(g)))
coverage_audited=pd.DataFrame(coverage_audited)
assert (coverage_audited[['explicit','broad_unclear','not_reported_NA']].sum(axis=1)==coverage_audited.denominator).all()
coverage_audited['explicit_percent']=(100*coverage_audited.explicit/coverage_audited.denominator).round(1)
display(coverage_audited)
print('Audited coverage supersedes the earlier automatic stage-information percentages.')


,country,stage,explicit,broad_unclear,not_reported_NA,denominator,explicit_percent
0,Nigeria,Symptom recognition,31,1,10,42,73.8
1,UK,Symptom recognition,129,30,41,200,64.5
2,Nigeria,Appraisal,10,10,22,42,23.8
3,UK,Appraisal,2,169,29,200,1.0
4,Nigeria,Initial action,20,5,17,42,47.6
5,UK,Initial action,132,37,31,200,66.0
6,Nigeria,First healthcare contact,26,7,9,42,61.9
7,UK,First healthcare contact,179,15,6,200,89.5
8,Nigeria,Referral initiated,11,5,26,42,26.2
9,UK,Referral initiated,137,40,23,200,68.5


Audited coverage supersedes the earlier automatic stage-information percentages.


### 3A. Nigerian regional coverage and pathway evidence

The 42 Nigerian publications were represented once in the study-level analysis. Multiregional publications could contribute to more than one region in the region-presence summaries. Consequently, regional-presence counts were not interpreted as additional independent studies or as comparisons of patient-level diagnostic delay.


In [ ]:
ng_metadata=s1[s1.analysis_country.eq('Nigeria')][
 ['analysis_study_id','nigeria_region_clean','cancer_type_clean','healthcare_setting_clean']].copy()
ng_s2=s2[s2.analysis_country.eq('Nigeria')].copy()
assert ng_s2.analysis_study_id.nunique()==42
rq1=ng_metadata.merge(ng_s2,on='analysis_study_id',validate='one_to_one')
assert len(rq1)==42
for stage,col in AUDIT_COLS.items():
    rq1[stage+'_evidence']=rq1[col].map(lambda v:clear_status(col,v))

# Show all study records with traceable original descriptions, plus standardized fields.
rq1_detail_columns=['analysis_study_id','nigeria_region_clean','cancer_type_clean','healthcare_setting_clean',
 'Pathway to care','Initial action','First healthcare contact','Referral initiated',
 'Diagnostic investigation','Diagnosis confirmation','Route to diagnosis']
assert all(c in rq1.columns for c in rq1_detail_columns)
display(rq1[rq1_detail_columns])

regional=rq1.assign(region=rq1.nigeria_region_clean.fillna('NR').astype(str).str.split(';')).explode('region')
regional['region']=regional.region.str.strip()
regional_counts=regional.groupby('region').agg(unique_studies=('analysis_study_id','nunique')).reset_index()
regional_counts=regional_counts.sort_values('unique_studies',ascending=False)
print('Regional presence — multiregional studies appear more than once, counts are NOT additive')
display(regional_counts)
print('Unique Nigeria studies:',rq1.analysis_study_id.nunique(),
      '| Region-presence sum:',regional_counts.unique_studies.sum())


,analysis_study_id,nigeria_region_clean,cancer_type_clean,healthcare_setting_clean,Pathway to care,Initial action,First healthcare contact,Referral initiated,Diagnostic investigation,Diagnosis confirmation,Route to diagnosis
0,NG01,national_or_multiregional,breast,healthcare_system_wide,Access to diagnostic imaging services varies g...,NR,NR,NR,"Mammography, breast ultrasound and ultrasound-...",Ultrasound-guided breast biopsy identified as ...,Health-system access to diagnostic imaging and...
1,NG02,south_west,gastrointestinal,tertiary_care,Symptom onset → initial action (often hospital...,Most patients visited a hospital after noticin...,Hospital; 55% initially presented to a private...,Patients were referred from other facilities t...,NR in sufficient detail for pathway mapping,Clinically and/or histologically confirmed GI ...,Tertiary-hospital diagnostic pathway; many pat...
2,NG03,north_central;south_west,breast,other_setting,Detection → healthcare-seeking pathway involvi...,Premorbid help-seeking preferences were assess...,Orthodox healthcare provider or alternative ca...,"Reasons for referral were collected, but refer...",NR as a detailed diagnostic sequence,Newly diagnosed breast cancer; detailed confir...,NaN
3,NG04,south_west,breast,tertiary_care,The study assessed trends in stage at diagnosi...,NR,NR,NR,NR,NR,NaN
4,NG05,north_central,lung,tertiary_care,Symptoms/signs → healthcare presentation → rad...,NR,NR,NR,Chest radiography 100%; chest CT 56.3%; pleura...,Tissue diagnosis/histopathological confirmation,Clinical presentation → chest radiography/CT →...
5,NG06,south_east,prostate,tertiary_care,Symptoms → hospital presentation → first visit...,NR,Hospital presentation/first visit,NR,NR,Histological confirmation/biopsy,Hospital-based diagnosis following presentatio...
6,NG07,south_west,all_cancers,community_based,Community perception → choice of cancer care →...,"Choice of cancer care influenced by sex, age, ...",Orthodox healthcare and other available health...,NR,NR,NR,Orthodox medical care was preferred/used by 60...
7,NG08,south_east,multiple_cancers,tertiary_care,Symptom onset → parent/caregiver decision and ...,Visits to health professionals and alternative...,"Various first facilities: mission hospital, pr...","Yes, for patients coming from non-cancer treat...",Diagnostic tests; histological/cytological ass...,Histological or cytological confirmation,"Routes included primary health care, private/g..."
8,NG09,north_west;south_west,multiple_cancers,tertiary_care,First symptom → appraisal/decision → possible ...,Some delayed formal care; others sought family...,Formal healthcare professional; participants c...,Failure to refer identified as an important so...,Diagnostic investigations culminating in histo...,Histological confirmation.,Patients moved through a median of 3 health fa...
9,NG10,south_west,ovarian,tertiary_care,Symptom onset → decision to seek healthcare → ...,Patients sought care at various facilities; 21...,"Private hospital (35.1%), herbal home (21.6%),...",Referral to LASUTH from initial healthcare fac...,Surgical evaluation/staging; authors note that...,Histological confirmation; surgical/histologic...,"Multiple routes including private hospitals, h..."


Regional presence — multiregional studies appear more than once, counts are NOT additive


,region,unique_studies
6,south_west,24
3,north_west,6
4,south_east,6
2,north_central,4
5,south_south,4
1,national_or_multiregional,2
0,NR,1


Unique Nigeria studies: 42 | Region-presence sum: 47


In [ ]:
# Region-by-stage evidence matrix: counts distinct studies WITH EXPLICITLY CLASSIFIED DATA.
rows=[]
for region,g in regional.groupby('region'):
    for stage in AUDIT_COLS:
        c=g[stage+'_evidence'].value_counts()
        rows.append({'region':region,'stage':stage,'studies_in_region':g.analysis_study_id.nunique(),
                     'explicit_studies':int(c.get('explicitly described',0)),
                     'broad_unclear_studies':int(c.get('broad / unclear',0)),
                     'not_reported_NA_studies':int(c.get('not reported / NA',0))})
region_stage_matrix=pd.DataFrame(rows)
display(region_stage_matrix)
print('These counts describe evidence coverage only, NOT the percentage of patients following a route.')


,region,stage,studies_in_region,explicit_studies,broad_unclear_studies,not_reported_NA_studies
0,NR,Symptom recognition,1,0,0,1
1,NR,Appraisal,1,0,0,1
2,NR,Initial action,1,1,0,0
3,NR,First healthcare contact,1,0,0,1
4,NR,Referral initiated,1,0,0,1
5,NR,Diagnostic investigation,1,0,0,1
6,NR,Diagnosis confirmation,1,0,0,1
7,NR,Route to diagnosis,1,0,0,1
8,national_or_multiregional,Symptom recognition,2,1,0,1
9,national_or_multiregional,Appraisal,2,0,0,2


These counts describe evidence coverage only, NOT the percentage of patients following a route.


### 3B. Routes, first healthcare contacts, referrals and diagnostic confirmation

Recorded categories for initial actions, first healthcare contact, referrals, diagnostic investigation and diagnosis confirmation were summarised at publication level. A traceable evidence register linked findings to their study identifiers. The analysis distinguished clearly reported pathway information from broad descriptions that did not establish an individual patient's complete journey.


In [ ]:
for col in ['pathway_to_care_clean','initial_action_clean','first_healthcare_contact_clean',
            'referral_initiated_clean','diagnostic_investigation_clean',
            'diagnosis_confirmation_clean','route_to_diagnosis_clean']:
    print('\n',col)
    display(rq1.groupby(col,dropna=False).analysis_study_id.nunique().reset_index(name='distinct_nigerian_studies')
            .sort_values('distinct_nigerian_studies',ascending=False))

# A transparent narrative worksheet for RQ1 — record direct observations rather than estimates.
rq1_narratives=rq1[['analysis_study_id','nigeria_region_clean','cancer_type_clean',
 'Pathway to care','First healthcare contact','Referral initiated','Diagnostic investigation',
 'Diagnosis confirmation','Route to diagnosis']].copy()
rq1_narratives['interpretation_note']=''  # fill manually for the studies selected for narrative synthesis
rq1_narratives['reporting_limitations']=''
display(rq1_narratives.head(12))



 pathway_to_care_clean


,pathway_to_care_clean,distinct_nigerian_studies
0,ordered_pathway_narrative,40
1,other_pathway_narrative,2



 initial_action_clean


,initial_action_clean,distinct_nigerian_studies
6,not_reported,17
5,multiple_formal_and_informal_actions,6
0,action_described_other_or_unclear,5
2,formal_healthcare_contact_or_referral,5
4,informal_or_alternative_care,4
3,help_seeking_unspecified,3
1,clinician_referral_or_pathway_event,2



 first_healthcare_contact_clean


,first_healthcare_contact_clean,distinct_nigerian_studies
2,hospital_or_formal_care_level_unspecified,13
5,not_reported,9
6,other_or_unclear,6
7,specialist_or_tertiary_care,6
3,multiple_or_variable_first_contacts,4
0,alternative_or_traditional_care,2
1,emergency_department,1
4,not_determined_from_description,1


... (5 outputs left out)

,diagnosis_confirmation_clean,distinct_nigerian_studies
6,pathology_or_cytology_reported,26
2,not_reported,5
1,narrative_or_unclear,4
4,pathology_and_clinical_reported,4
0,clinical_confirmation_reported,1
3,not_reported_with_context,1
5,pathology_and_imaging_reported,1



 route_to_diagnosis_clean


,route_to_diagnosis_clean,distinct_nigerian_studies
1,narrative_or_unclear,19
3,other_clinical_route_mentioned,17
0,blank,2
2,not_reported,2
4,primary_care_route_mentioned,1
5,screening_route_mentioned,1


,analysis_study_id,nigeria_region_clean,cancer_type_clean,Pathway to care,First healthcare contact,Referral initiated,Diagnostic investigation,Diagnosis confirmation,Route to diagnosis,interpretation_note,reporting_limitations
0,NG01,national_or_multiregional,breast,Access to diagnostic imaging services varies g...,NR,NR,"Mammography, breast ultrasound and ultrasound-...",Ultrasound-guided breast biopsy identified as ...,Health-system access to diagnostic imaging and...,,
1,NG02,south_west,gastrointestinal,Symptom onset → initial action (often hospital...,Hospital; 55% initially presented to a private...,Patients were referred from other facilities t...,NR in sufficient detail for pathway mapping,Clinically and/or histologically confirmed GI ...,Tertiary-hospital diagnostic pathway; many pat...,,
2,NG03,north_central;south_west,breast,Detection → healthcare-seeking pathway involvi...,Orthodox healthcare provider or alternative ca...,"Reasons for referral were collected, but refer...",NR as a detailed diagnostic sequence,Newly diagnosed breast cancer; detailed confir...,NaN,,
3,NG04,south_west,breast,The study assessed trends in stage at diagnosi...,NR,NR,NR,NR,NaN,,
4,NG05,north_central,lung,Symptoms/signs → healthcare presentation → rad...,NR,NR,Chest radiography 100%; chest CT 56.3%; pleura...,Tissue diagnosis/histopathological confirmation,Clinical presentation → chest radiography/CT →...,,
5,NG06,south_east,prostate,Symptoms → hospital presentation → first visit...,Hospital presentation/first visit,NR,NR,Histological confirmation/biopsy,Hospital-based diagnosis following presentatio...,,
6,NG07,south_west,all_cancers,Community perception → choice of cancer care →...,Orthodox healthcare and other available health...,NR,NR,NR,Orthodox medical care was preferred/used by 60...,,
7,NG08,south_east,multiple_cancers,Symptom onset → parent/caregiver decision and ...,"Various first facilities: mission hospital, pr...","Yes, for patients coming from non-cancer treat...",Diagnostic tests; histological/cytological ass...,Histological or cytological confirmation,"Routes included primary health care, private/g...",,
8,NG09,north_west;south_west,multiple_cancers,First symptom → appraisal/decision → possible ...,Formal healthcare professional; participants c...,Failure to refer identified as an important so...,Diagnostic investigations culminating in histo...,Histological confirmation.,Patients moved through a median of 3 health fa...,,
9,NG10,south_west,ovarian,Symptom onset → decision to seek healthcare → ...,"Private hospital (35.1%), herbal home (21.6%),...",Referral to LASUTH from initial healthcare fac...,Surgical evaluation/staging; authors note that...,Histological confirmation; surgical/histologic...,"Multiple routes including private hospitals, h...",,


## 4. Thematic coding and framework mapping

The approved coding framework was applied to **979 factor-level findings** in Sheet 3 and **472 problem-level findings** in Sheet 4. Original factor descriptions and supporting evidence were retained alongside the resulting coded fields.

The initial automated mapping produced a transparent classification record. Categories that could not be mapped reliably were marked for review. The earlier 20-finding coding pilot was preserved as a distinct preceding methodological stage rather than repeated in this notebook.


In [ ]:
theme_table=pd.read_excel(PATHS['codebook'],sheet_name='Themes')
map3=pd.read_excel(PATHS['codebook'],sheet_name='Sheet 3 Map')
map4=pd.read_excel(PATHS['codebook'],sheet_name='Sheet 4 Map')
intervention_table=pd.read_excel(PATHS['codebook'],sheet_name='Interventions')

def key(v):return re.sub(r'\s+',' ',str(v).strip()).casefold()
def build_map(mapping):
    result={}
    for _,r in mapping.iterrows():
        if pd.isna(r.iloc[0]) or pd.isna(r.iloc[1]):continue
        result[key(r.iloc[0])]=str(r.iloc[1]).strip().upper()
    return result

MAP3=build_map(map3);MAP4=build_map(map4)
valid_codes=set(theme_table['Code'].dropna().astype(str).str.strip())
print('Approved themes:',sorted(valid_codes))
print('Mapped Sheet 3 categories:', len(MAP3), '| Sheet 4 categories:',len(MAP4))
assert all(set(re.findall(r'T\d+',v))<=valid_codes for v in list(MAP3.values())+list(MAP4.values()))

def assign_theme(value,mapping):
    if pd.isna(value) or not str(value).strip():return ('','REVIEW','Missing category')
    parts=[key(s) for s in str(value).split(';') if s.strip()]
    matches=[];unmatched=[]
    for part in parts:
        mapped=mapping.get(part)
        if mapped:
            matches.extend(re.findall(r'T\d+',mapped))
        else:unmatched.append(part)
    matches=sorted(set(matches),key=lambda x:int(x[1:]))
    if unmatched:
        return ('; '.join(matches),'REVIEW','Unmapped category: '+'; '.join(unmatched))
    if not matches:return ('','REVIEW','No valid theme code')
    return ('; '.join(matches),'MAPPED_BY_RULE','Exact framework category mapping')

for df,category,mapping in [(s3,'factor_category_clean',MAP3),
                             (s4,'Problem category (clean)',MAP4)]:
    assert category in df.columns, f'Missing classification column {category}'
    coded=df[category].apply(lambda x:pd.Series(assign_theme(x,mapping),
                                      index=['suggested_theme_codes','coding_status','coding_reason']))
    for col in coded.columns:df[col]=coded[col].to_numpy()

s3['source_sheet']='Sheet 3'
s4['source_sheet']='Sheet 4'
s3['source_row_excel']=s3.index+2
s4['source_row_excel']=s4.index+2
# Retain original source columns. Sheet 4 has no directly equivalent standardized pathway/effect-direction fields.
s4['pathway_stage_primary']='Not separately coded in Sheet 4'
s4['effect_direction_clean']='Not separately coded in Sheet 4'

print('Sheet 3 coding statuses:');display(s3.coding_status.value_counts().rename_axis('status').reset_index(name='finding_rows'))
print('Sheet 4 coding statuses:');display(s4.coding_status.value_counts().rename_axis('status').reset_index(name='finding_rows'))
assert len(s3)+len(s4)>0

Approved themes: ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8']
Mapped Sheet 3 categories: 12 | Sheet 4 categories: 11
Sheet 3 coding statuses:


,status,finding_rows
0,MAPPED_BY_RULE,911
1,REVIEW,68


Sheet 4 coding statuses:


,status,finding_rows
0,MAPPED_BY_RULE,472


### 4A. Review of initially unmatched categories

The first coding pass identified categories that were not resolved by the automated framework. Their original `coding_status` values were retained as part of the audit trail. Subsequent researcher-approved decisions were recorded in separate final coding fields, allowing initial machine-generated flags to be distinguished from the accepted classifications.


In [ ]:
review3=s3[s3.coding_status.eq('REVIEW')].copy()
review4=s4[s4.coding_status.eq('REVIEW')].copy()
review=pd.concat([
    review3[['source_sheet','source_row_excel','analysis_study_id','factor_category_clean',
             'factor_subcategory_clean','suggested_theme_codes','coding_reason']].rename(columns={
                 'factor_category_clean':'existing_category','factor_subcategory_clean':'existing_subcategory'}),
    review4[['source_sheet','source_row_excel','analysis_study_id','Problem category (clean)',
             'Problem subcategory (clean)','suggested_theme_codes','coding_reason']].rename(columns={
                 'Problem category (clean)':'existing_category','Problem subcategory (clean)':'existing_subcategory'})
],ignore_index=True)
review['review_decision']=''
review['final_theme_codes']=''
review['review_rationale']=''
print('Unmapped findings requiring decisions:',len(review))
display(review.head(25))
print('Most common unmapped categories:')
display(review.groupby(['source_sheet','existing_category'],dropna=False).size().reset_index(name='finding_rows').sort_values('finding_rows',ascending=False).head(30))

Unmapped findings requiring decisions: 68


,source_sheet,source_row_excel,analysis_study_id,existing_category,existing_subcategory,suggested_theme_codes,coding_reason,review_decision,final_theme_codes,review_rationale
0,Sheet 3,36,NG07,Other or mixed,Review original description,,No valid theme code,,,
1,Sheet 3,38,NG07,Other or mixed,Review original description,,No valid theme code,,,
2,Sheet 3,97,NG12,Other or mixed,Review original description,,No valid theme code,,,
3,Sheet 3,161,NG15,Other or mixed,Review original description,,No valid theme code,,,
4,Sheet 3,172,NG16,Other or mixed,Review original description,,No valid theme code,,,
5,Sheet 3,261,NG24,Other or mixed,Review original description,,No valid theme code,,,
6,Sheet 3,279,NG26,Other or mixed,Review original description,,No valid theme code,,,
7,Sheet 3,362,NG34,Other or mixed,Review original description,,No valid theme code,,,
8,Sheet 3,363,NG34,Other or mixed,Review original description,,No valid theme code,,,
9,Sheet 3,371,NG35,Other or mixed,Review original description,,No valid theme code,,,


Most common unmapped categories:


,source_sheet,existing_category,finding_rows
0,Sheet 3,Other or mixed,67
1,Sheet 3,Unclassified,1


## 5. Researcher-approved thematic decisions

A **68-item decision register** documented the resolution of flagged Sheet 3 findings. Each decision was linked to its original Excel row, study identifier and extracted factor wording to protect against mismatched assignments.

On **9 November 2026**, 61 findings were assigned approved thematic codes and seven were retained as unclassified. Decision rationales and original evidence wording were preserved. The accepted classifications were applied to working analysis tables without overwriting the source workbooks.


In [ ]:
# Embedded, inspectable decision register (68 exact Excel rows from Sheet 3).
DECISIONS = {'36': {'study_id': 'NG07', 'specific_factor': 'Problems encountered at healthcare institutions', 'proposed_theme_codes': 'T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Institution-level problems discussed, but no significant association was reported; not a demonstrated delay barrier.'}, '38': {'study_id': 'NG07', 'specific_factor': 'Perceived benefits of orthodox care', 'proposed_theme_codes': 'T1', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Preference for orthodox care concerns healthcare choice; observational preference, not a measured delay.'}, '97': {'study_id': 'NG12', 'specific_factor': 'Alternative practitioners/prayer houses as initial treatment', 'proposed_theme_codes': 'T1; T2', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Initial alternative/prayer-house care concerns help-seeking and beliefs; association with delay reported.'}, '161': {'study_id': 'NG15', 'specific_factor': 'Interaction between ecological factors', 'proposed_theme_codes': 'UNCLASSIFIED', 'review_decision': 'RETAIN_UNCLASSIFIED', 'rationale': 'Interactions span several levels without a separable factor; retain mixed and do not force an individual theme.'}, '172': {'study_id': 'NG16', 'specific_factor': 'Opportunistic rather than organised screening', 'proposed_theme_codes': 'T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Opportunistic rather than organised detection concerns diagnostic/screening routes; palliative assignment is not itself a diagnostic-delay estimate.'}, '261': {'study_id': 'NG24', 'specific_factor': 'Employment status', 'proposed_theme_codes': 'T2', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Employment status is socioeconomic; study reports no significant contribution to delay.'}, '279': {'study_id': 'NG26', 'specific_factor': 'Outdoor occupation/high sun exposure', 'proposed_theme_codes': 'T7', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Sun exposure is a clinical risk context, not evidence of diagnostic delay.'}, '362': {'study_id': 'NG34', 'specific_factor': 'Maternal age', 'proposed_theme_codes': 'T2', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Maternal age is a patient/family sociodemographic correlate; shorter lag reported, not necessarily causal.'}, '363': {'study_id': 'NG34', 'specific_factor': 'Age at diagnosis', 'proposed_theme_codes': 'T7', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Age at diagnosis is a clinical/patient characteristic associated with lag; no causal claim.'}, '371': {'study_id': 'NG35', 'specific_factor': 'Use of alternative medicine/traditional healers', 'proposed_theme_codes': 'T1', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Alternative healers are a help-seeking pathway; retain study-specific strength of evidence.'}, '394': {'study_id': 'NG37', 'specific_factor': 'Pregnancy status masking breast cancer symptoms', 'proposed_theme_codes': 'T7; T1', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Pregnancy-associated symptom masking and delayed recognition involve clinical context and appraisal.'}, '395': {'study_id': 'NG37', 'specific_factor': 'Prayer-house use', 'proposed_theme_codes': 'T1; T2', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Prayer-house use concerns informal help-seeking and social/religious practices.'}, '425': {'study_id': 'NG40', 'specific_factor': 'Fear of cancer', 'proposed_theme_codes': 'T1', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Fear of cancer affects appraisal and help-seeking; barrier reported by patients.'}, '426': {'study_id': 'NG40', 'specific_factor': 'Fear of surgery', 'proposed_theme_codes': 'T1', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Fear of surgery affects anticipated care and help-seeking; barrier reported by patients.'}, '536': {'study_id': 'UK011', 'specific_factor': 'Emergency presentation', 'proposed_theme_codes': 'T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Emergency presentation is a route to diagnosis; survival association does not prove delay.'}, '555': {'study_id': 'UK018', 'specific_factor': 'Need to track referral receipt and appointments', 'proposed_theme_codes': 'T4; T8', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Tracking referral receipt and appointments concerns referral coordination and information transfer; process description.'}, '566': {'study_id': 'UK022', 'specific_factor': 'Hospital Trust', 'proposed_theme_codes': 'T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Variation between hospital trusts concerns organisational context, not automatically a causal barrier.'}, '567': {'study_id': 'UK022', 'specific_factor': 'Local diagnostic service provision/culture', 'proposed_theme_codes': 'T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Local diagnostic services and culture concern service organisation; recommendation only.'}, '577': {'study_id': 'UK025', 'specific_factor': 'Higher proportion of current smokers', 'proposed_theme_codes': 'T7', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Smoking is a risk/clinical characteristic; direction for diagnostic timeliness not established.'}, '578': {'study_id': 'UK025', 'specific_factor': 'Higher frequency of seeing preferred GP', 'proposed_theme_codes': 'T3', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Seeing preferred GP relates to continuity of care; referral-rate association is not equivalent to delay.'}, '579': {'study_id': 'UK025', 'specific_factor': 'Patients feeling involved in care decisions', 'proposed_theme_codes': 'T8', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Patient involvement in care decisions concerns communication; referral predictor, not demonstrated delay.'}, '586': {'study_id': 'UK026', 'specific_factor': 'Urban residence', 'proposed_theme_codes': 'T6', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Urban residence is geographic context affecting referral patterns, not proven delay.'}, '590': {'study_id': 'UK028', 'specific_factor': 'Variation between general practices', 'proposed_theme_codes': 'T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Practice-level variation concerns service organisation; outcome-specific relevance requires care.'}, '591': {'study_id': 'UK028', 'specific_factor': 'Chance variation', 'proposed_theme_codes': 'UNCLASSIFIED', 'review_decision': 'RETAIN_UNCLASSIFIED', 'rationale': 'Chance variation is statistical uncertainty, not an actionable patient/provider/system factor.'}, '592': {'study_id': 'UK028', 'specific_factor': 'Emergency-presentation outcome indicators', 'proposed_theme_codes': 'UNCLASSIFIED', 'review_decision': 'RETAIN_UNCLASSIFIED', 'rationale': 'Outcome indicator reliability is methodological, not a documented diagnostic barrier.'}, '595': {'study_id': 'UK029', 'specific_factor': 'Need for clear LHW role boundaries', 'proposed_theme_codes': 'T5; T8', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Lay health worker roles involve workforce organisation and communication; stakeholder preference.'}, '599': {'study_id': 'UK030', 'specific_factor': 'Multivariable model', 'proposed_theme_codes': 'UNCLASSIFIED', 'review_decision': 'RETAIN_UNCLASSIFIED', 'rationale': 'Model AUC is predictive performance, not a pathway barrier/facilitator in itself.'}, '607': {'study_id': 'UK032', 'specific_factor': 'Symptom-based risk calculator', 'proposed_theme_codes': 'T3', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Risk calculator is a proposed clinical decision-support aid; diagnostic effect not evaluated.'}, '623': {'study_id': 'UK039', 'specific_factor': 'Machine-learning classification', 'proposed_theme_codes': 'T3', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Machine-learning classification is a diagnostic decision-support approach; performance is not proof of shorter diagnosis.'}, '626': {'study_id': 'UK040', 'specific_factor': 'Higher alcohol consumption', 'proposed_theme_codes': 'T7', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Alcohol consumption relates to cancer risk, not demonstrated diagnostic delay.'}, '633': {'study_id': 'UK042', 'specific_factor': 'Low awareness of HaNC-RC V2', 'proposed_theme_codes': 'T3', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Unfamiliarity with risk calculator concerns clinician/tool awareness; do not assume delayed diagnosis.'}, '634': {'study_id': 'UK043', 'specific_factor': 'Waiting between interventions', 'proposed_theme_codes': 'T1', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Patient concern about waiting is an appraisal/perception; no measured waiting effect.'}, '638': {'study_id': 'UK044', 'specific_factor': 'Lack of trust in healthcare providers', 'proposed_theme_codes': 'T1; T8', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Trust in healthcare services can influence help-seeking and communication; association not quantified.'}, '639': {'study_id': 'UK044', 'specific_factor': 'Experiences of racism and negative healthcare experiences', 'proposed_theme_codes': 'T2; T8', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Racism and negative care experiences concern social inequity and interpersonal communication.'}, '640': {'study_id': 'UK045', 'specific_factor': '2WW referral rate', 'proposed_theme_codes': 'T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Urgent referral rate is a referral-process indicator associated with stage; not necessarily delay.'}, '641': {'study_id': 'UK045', 'specific_factor': 'TWW detection rate', 'proposed_theme_codes': 'T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Detection rate through urgent route is a pathway performance indicator; do not generalise causality.'}, '649': {'study_id': 'UK048', 'specific_factor': 'Increased colonoscopy/testing', 'proposed_theme_codes': 'UNCLASSIFIED', 'review_decision': 'RETAIN_UNCLASSIFIED', 'rationale': 'Testing costs alone are economic evaluation outcomes, not established barriers to diagnostic timeliness.'}, '650': {'study_id': 'UK049', 'specific_factor': 'Non-compliant primary-care referrals', 'proposed_theme_codes': 'T3; T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Referral non-compliance concerns provider practice and guideline-based referral; preserve original effect direction.'}, '653': {'study_id': 'UK050', 'specific_factor': 'Larger practices', 'proposed_theme_codes': 'T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Practice size is organisational context; association outcome should be specified before asserting delay.'}, '655': {'study_id': 'UK050', 'specific_factor': 'More deprived practice populations', 'proposed_theme_codes': 'T2', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Practice population deprivation is socioeconomic context associated with detection.'}, '658': {'study_id': 'UK051', 'specific_factor': 'Insufficient precision in identifying high-risk patients', 'proposed_theme_codes': 'T3', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Risk stratification relates to provider diagnostic decisions; recommendation, not evaluated effect.'}, '669': {'study_id': 'UK058', 'specific_factor': 'Training-practice status', 'proposed_theme_codes': 'T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Training practice status is service/workforce organisation linked to urgent referral rates.'}, '682': {'study_id': 'UK066', 'specific_factor': 'Booking-system difficulties', 'proposed_theme_codes': 'T4; T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Booking-system difficulties affect referral logistics and service organisation.'}, '688': {'study_id': 'UK069', 'specific_factor': 'Nonspecific symptoms', 'proposed_theme_codes': 'T7; T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Non-specific symptoms are clinical presentation interacting with fast-track referral decisions.'}, '691': {'study_id': 'UK071', 'specific_factor': 'Nonmedical practitioner referral', 'proposed_theme_codes': 'T3; T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Referral source is a provider/referral route factor; observed malignancy proportion, not delay.'}, '699': {'study_id': 'UK076', 'specific_factor': 'Not knowing who to ask for information', 'proposed_theme_codes': 'T8', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Uncertainty over who to ask is an information/navigation barrier.'}, '710': {'study_id': 'UK084', 'specific_factor': 'Evidence-to-practice gap', 'proposed_theme_codes': 'T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Evidence-to-practice translation is a health-system implementation issue; consensus observation.'}, '720': {'study_id': 'UK087', 'specific_factor': 'Diagnostic-process inefficiencies', 'proposed_theme_codes': 'T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Diagnostic-process inefficiencies concern investigation pathway design; review of technologies is not proof of effect.'}, '731': {'study_id': 'UK091', 'specific_factor': 'Incomplete safety-netting', 'proposed_theme_codes': 'T3; T8', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Safety-netting documentation concerns clinical follow-up and communication; frequency only.'}, '732': {'study_id': 'UK092', 'specific_factor': 'Delayed identification/prioritisation of suspicious CXR findings', 'proposed_theme_codes': 'T3; T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'AI prioritisation of abnormal radiographs is diagnostic triage; trial assessment, effect not assumed.'}, '760': {'study_id': 'UK102', 'specific_factor': 'Ambiguous guidance', 'proposed_theme_codes': 'T3', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Ambiguous clinical guidance affects professional decision support; evidence quality stated.'}, '764': {'study_id': 'UK103', 'specific_factor': 'Prison control/custody restrictions', 'proposed_theme_codes': 'T2; T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Custody/prison regime creates social/institutional access constraints; preserve context.'}, '766': {'study_id': 'UK104', 'specific_factor': 'Neoadjuvant chemotherapy', 'proposed_theme_codes': 'UNCLASSIFIED', 'review_decision': 'RETAIN_UNCLASSIFIED', 'rationale': 'Neoadjuvant chemotherapy concerns treatment planning/timing; do not interpret as pre-diagnostic delay.'}, '798': {'study_id': 'UK118', 'specific_factor': 'Limited diagnostic resources', 'proposed_theme_codes': 'T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Diagnostic resources are a service-capacity constraint; study describes increased delay.'}, '800': {'study_id': 'UK119', 'specific_factor': 'Less common tumour types', 'proposed_theme_codes': 'T7', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Tumour-type diversity is clinical case mix, not itself evidence of delay.'}, '804': {'study_id': 'UK121', 'specific_factor': 'Multiple GP consultations', 'proposed_theme_codes': 'T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Repeated GP consultations are a diagnostic-pathway process; association with delay must be study-specific.'}, '833': {'study_id': 'UK131', 'specific_factor': 'Greater deprivation', 'proposed_theme_codes': 'T2; T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Deprivation associated with emergency diagnostic route; association is not causal.'}, '855': {'study_id': 'UK140', 'specific_factor': 'Being too busy', 'proposed_theme_codes': 'T1', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Being too busy is a stated help-seeking barrier.'}, '856': {'study_id': 'UK140', 'specific_factor': 'Embarrassment', 'proposed_theme_codes': 'T1; T2', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Embarrassment is a psychological/social barrier to presenting.'}, '857': {'study_id': 'UK141', 'specific_factor': 'Patient-doctor continuity', 'proposed_theme_codes': 'T3', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Patient-doctor continuity is a provider relationship factor with cancer-specific association.'}, '863': {'study_id': 'UK143', 'specific_factor': 'Limited safety-netting documentation', 'proposed_theme_codes': 'T3; T8', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Safety-netting documentation concerns provider follow-up and information exchange.'}, '866': {'study_id': 'UK144', 'specific_factor': 'A&E attendance', 'proposed_theme_codes': 'T4', 'review_decision': 'PROPOSED_CODE', 'rationale': 'A&E attendance is a route/contact measure; association with long intervals is not causal.'}, '871': {'study_id': 'UK145', 'specific_factor': 'Lack of lead clinician', 'proposed_theme_codes': 'T4; T8', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Lack of lead clinician may undermine care coordination and communication; participant-reported.'}, '877': {'study_id': 'UK147', 'specific_factor': 'Lack of integrated laboratory-haematology pathway', 'proposed_theme_codes': 'T4; T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Lack of laboratory-haematology integration concerns pathway coordination and service organisation.'}, '946': {'study_id': 'UK185', 'specific_factor': 'Prison healthcare setting', 'proposed_theme_codes': 'T2; T5', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Prison-based care is a socially distinct setting with service organisation constraints; findings needed for effect.'}, '966': {'study_id': 'UK191', 'specific_factor': 'Road infrastructure', 'proposed_theme_codes': 'T6', 'review_decision': 'PROPOSED_CODE', 'rationale': 'Road infrastructure concerns geographic and transport access.'}, '975': {'study_id': 'UK197', 'specific_factor': 'GP recognition/management', 'proposed_theme_codes': 'T3', 'review_decision': 'PROPOSED_CODE', 'rationale': 'GP recognition and management concerns provider clinical practice; secondary review synthesis.'}, '976': {'study_id': 'UK198', 'specific_factor': 'No results available', 'proposed_theme_codes': 'UNCLASSIFIED', 'review_decision': 'RETAIN_UNCLASSIFIED', 'rationale': 'Study is protocol-only with no reported findings; retain uncoded rather than infer evidence.'}}
assert len(DECISIONS)==68
review68=s3[s3.coding_status.eq('REVIEW')].copy()
assert len(review68)==68
review68['source_row_excel']=review68.index+2
assert set(review68.source_row_excel.astype(str))==set(DECISIONS)

register=[]
for _,r in review68.iterrows():
    row_num=str(int(r.source_row_excel)); d=DECISIONS[row_num]
    assert str(r.analysis_study_id)==d['study_id']
    assert str(r['Specific factor original']).strip()==d['specific_factor'].strip(), f'Source changed at Excel row {row_num}'
    register.append({'source_excel_row':int(row_num),'study_id':d['study_id'],
     'specific_factor':d['specific_factor'], 'evidence_finding':r['Evidence/Finding original'],
     'effect_direction_clean':r['effect_direction_clean'],
     'proposed_theme_codes':d['proposed_theme_codes'],
     'review_decision':d['review_decision'], 'rationale':d['rationale'],
     'researcher_approval':'APPROVED','final_theme_codes':(d['proposed_theme_codes'] if d['review_decision']=='PROPOSED_CODE' else ''),'researcher_notes':'Approved by researcher 2026-11-09; retained unclassified where indicated.'})
decision_register=pd.DataFrame(register)
print('Previously flagged decisions, all approved by researcher on 2026-11-09:')
display(decision_register.review_decision.value_counts())
display(decision_register)


Previously flagged decisions, all approved by researcher on 2026-11-09:


,count
review_decision,
PROPOSED_CODE,61
RETAIN_UNCLASSIFIED,7


,source_excel_row,study_id,specific_factor,evidence_finding,effect_direction_clean,proposed_theme_codes,review_decision,rationale,researcher_approval,final_theme_codes,researcher_notes
0,36,NG07,Problems encountered at healthcare institutions,Problems encountered at healthcare institution...,direction_not_clear,T5,PROPOSED_CODE,"Institution-level problems discussed, but no s...",APPROVED,T5,Approved by researcher 2026-11-09; retained un...
1,38,NG07,Perceived benefits of orthodox care,"60% preferred and used orthodox medical care, ...",direction_not_clear,T1,PROPOSED_CODE,Preference for orthodox care concerns healthca...,APPROVED,T1,Approved by researcher 2026-11-09; retained un...
2,97,NG12,Alternative practitioners/prayer houses as ini...,21/27 delayed patients versus 36/91 non-delaye...,delaying_or_adverse,T1; T2,PROPOSED_CODE,Initial alternative/prayer-house care concerns...,APPROVED,T1; T2,Approved by researcher 2026-11-09; retained un...
3,161,NG15,Interaction between ecological factors,"Authors reported that intrapersonal, interpers...",direction_not_clear,UNCLASSIFIED,RETAIN_UNCLASSIFIED,Interactions span several levels without a sep...,APPROVED,,Approved by researcher 2026-11-09; retained un...
4,172,NG16,Opportunistic rather than organised screening,"Health-check-up detection was common, but thes...",direction_not_clear,T4,PROPOSED_CODE,Opportunistic rather than organised detection ...,APPROVED,T4,Approved by researcher 2026-11-09; retained un...
...,...,...,...,...,...,...,...,...,...,...,...
63,877,UK147,Lack of integrated laboratory-haematology pathway,Authors advocate integration,delaying_or_adverse,T4; T5,PROPOSED_CODE,Lack of laboratory-haematology integration con...,APPROVED,T4; T5,Approved by researcher 2026-11-09; retained un...
64,946,UK185,Prison healthcare setting,Study examined diagnosis and treatment within ...,delaying_or_adverse,T2; T5,PROPOSED_CODE,Prison-based care is a socially distinct setti...,APPROVED,T2; T5,Approved by researcher 2026-11-09; retained un...
65,966,UK191,Road infrastructure,Identified as a contextual barrier,delaying_or_adverse,T6,PROPOSED_CODE,Road infrastructure concerns geographic and tr...,APPROVED,T6,Approved by researcher 2026-11-09; retained un...
66,975,UK197,GP recognition/management,Review synthesised primary-care factors affect...,delaying_or_adverse,T3,PROPOSED_CODE,GP recognition and management concerns provide...,APPROVED,T3,Approved by researcher 2026-11-09; retained un...


In [ ]:
# Researcher approval logged: 2026-11-09. This cell is the reproducible approval step.
assert len(decision_register) == 68
assert decision_register['researcher_approval'].eq('APPROVED').all()
assert decision_register['review_decision'].eq('PROPOSED_CODE').sum() == 61
assert decision_register['review_decision'].eq('RETAIN_UNCLASSIFIED').sum() == 7

# The approval is recorded in the register above; this step applies it to in-memory analysis data.
# Apply approved decisions only to the working analysis DataFrame (s3), not the cleaned Excel file.
s3['final_theme_codes'] = s3['suggested_theme_codes'].fillna('')
s3['final_coding_status'] = s3['coding_status']
s4['final_theme_codes'] = s4['suggested_theme_codes'].fillna('')
s4['final_coding_status'] = s4['coding_status']
for _, row in decision_register.iterrows():
    mask = s3['source_row_excel'].eq(row['source_excel_row'])
    assert mask.sum() == 1, f"Expected exactly one source row: {row['source_excel_row']}"
    record=s3.loc[mask].iloc[0]
    assert str(record['analysis_study_id']) == str(row['study_id'])
    assert str(record['Specific factor original']).strip() == str(row['specific_factor']).strip()
    s3.loc[mask, 'final_theme_codes'] = row['final_theme_codes']
    s3.loc[mask, 'final_coding_status'] = (
        'APPROVED_MANUAL_CODE' if row['review_decision']=='PROPOSED_CODE'
        else 'APPROVED_UNCLASSIFIED'
    )

assert s3.final_coding_status.eq('APPROVED_MANUAL_CODE').sum()==61
assert s3.final_coding_status.eq('APPROVED_UNCLASSIFIED').sum()==7
assert s3.final_coding_status.eq('REVIEW').sum()==0
assert s4.final_coding_status.eq('REVIEW').sum()==0

final_parts=[]
for df in (s3,s4):
    x=df[['analysis_study_id','analysis_country','source_sheet','final_theme_codes']].copy()
    x['theme_code']=x.final_theme_codes.fillna('').str.split(';')
    x=x.explode('theme_code')
    x['theme_code']=x['theme_code'].astype(str).str.strip()
    final_parts.append(x[x.theme_code.isin(valid_codes)])
final_longthemes=pd.concat(final_parts,ignore_index=True)
final_theme_counts=(final_longthemes.groupby(['source_sheet','analysis_country','theme_code'])
 .agg(finding_rows=('analysis_study_id','size'),unique_studies=('analysis_study_id','nunique'))
 .reset_index()
 .merge(theme_table[['Code','Main theme']],left_on='theme_code',right_on='Code',how='left')
 .drop(columns='Code'))
print('Researcher-approved codes applied: 61; approved unclassified rows: 7')
print('Final themes counted as finding rows AND distinct studies separately.')
display(final_theme_counts)


Researcher-approved codes applied: 61; approved unclassified rows: 7
Final themes counted as finding rows AND distinct studies separately.


,source_sheet,analysis_country,theme_code,finding_rows,unique_studies,Main theme
0,Sheet 3,Nigeria,T1,44,15,Patient awareness and help-seeking
1,Sheet 3,Nigeria,T2,198,38,Social and financial factors
2,Sheet 3,Nigeria,T3,37,17,Healthcare-provider factors
3,Sheet 3,Nigeria,T4,2,2,Referral and diagnostic processes
4,Sheet 3,Nigeria,T5,129,36,Health-system and diagnostic capacity
5,Sheet 3,Nigeria,T6,22,16,Geographic accessibility
6,Sheet 3,Nigeria,T7,24,17,Clinical and disease-related factors
7,Sheet 3,Nigeria,T8,2,1,Communication and information
8,Sheet 3,UK,T1,46,37,Patient awareness and help-seeking
9,Sheet 3,UK,T2,87,52,Social and financial factors


### 5A. Validation of final thematic classifications

Validation checks confirmed the linkage between each approved decision and its original Sheet 3 entry, the consistency of approval statuses, and the use of codes from the approved thematic framework. Seven findings remained intentionally unclassified. Counts of coded rows were distinguished from counts of individual theme assignments because a finding could receive multiple themes.


In [ ]:
# Fully reproducible final coding assertions
assert len(decision_register) == 68
assert decision_register['source_excel_row'].is_unique
assert decision_register['researcher_approval'].eq('APPROVED').all()
assert decision_register['review_decision'].eq('PROPOSED_CODE').sum() == 61
assert decision_register['review_decision'].eq('RETAIN_UNCLASSIFIED').sum() == 7

assert len(s3) == 979 and len(s4) == 472
assert s3['final_coding_status'].eq('APPROVED_MANUAL_CODE').sum() == 61
assert s3['final_coding_status'].eq('APPROVED_UNCLASSIFIED').sum() == 7
assert not s3['final_coding_status'].eq('REVIEW').any()
assert not s4['final_coding_status'].eq('REVIEW').any()

all_findings = pd.concat([s3[['final_theme_codes']], s4[['final_theme_codes']]],ignore_index=True)
actual_codes = set()
for value in all_findings['final_theme_codes'].dropna().astype(str):
    actual_codes.update(token.strip() for token in value.split(';') if token.strip())
assert actual_codes.issubset(valid_codes), f'Unknown final themes: {actual_codes - valid_codes}'

coded_rows = all_findings['final_theme_codes'].fillna('').astype(str).str.strip().ne('').sum()
assert coded_rows == 1444, f'Expected 1,444 coded finding rows, found {coded_rows}'
assert len(all_findings) - coded_rows == 7
assert final_theme_counts['finding_rows'].sum() >= coded_rows, 'Multi-coded findings must count within each assigned theme'
print('CHECKS PASSED: 1,451 finding rows | 1,444 with themes | 7 unclassified | 0 unresolved')
print('Study-level country totals: Nigeria 42; UK 200; all four source datasets matched.')
print('Final, approved theme counts (do NOT use preliminary automated frequencies):')
display(final_theme_counts)


CHECKS PASSED: 1,451 finding rows | 1,444 with themes | 7 unclassified | 0 unresolved
Study-level country totals: Nigeria 42; UK 200; all four source datasets matched.
Final, approved theme counts (do NOT use preliminary automated frequencies):


,source_sheet,analysis_country,theme_code,finding_rows,unique_studies,Main theme
0,Sheet 3,Nigeria,T1,44,15,Patient awareness and help-seeking
1,Sheet 3,Nigeria,T2,198,38,Social and financial factors
2,Sheet 3,Nigeria,T3,37,17,Healthcare-provider factors
3,Sheet 3,Nigeria,T4,2,2,Referral and diagnostic processes
4,Sheet 3,Nigeria,T5,129,36,Health-system and diagnostic capacity
5,Sheet 3,Nigeria,T6,22,16,Geographic accessibility
6,Sheet 3,Nigeria,T7,24,17,Clinical and disease-related factors
7,Sheet 3,Nigeria,T8,2,1,Communication and information
8,Sheet 3,UK,T1,46,37,Patient awareness and help-seeking
9,Sheet 3,UK,T2,87,52,Social and financial factors


## 6. Register of extracted intervention evidence

The Sheet 4 register retained descriptions of intervention proposals, reported actions, evidence of effectiveness and potential applicability to Nigeria. These extracted fields were preserved without treating recommendations or unevaluated interventions as demonstrated effective.


In [ ]:
ivcols=['analysis_study_id','analysis_country','source_row_excel','Intervention / solution ',
        'Intervention level (clean)','Evidence of effectiveness ','Potential applicability to Nigeria',
        'Adaptation required','Key lesson']
missing=[c for c in ivcols if c not in s4.columns]
if missing: print('Missing optional intervention columns:',missing)
intervention_evidence=s4[[c for c in ivcols if c in s4.columns]].copy()
display(intervention_table)
display(intervention_evidence.head(8))

,Code,Intervention type,Coding rule
0,I1,Public awareness and patient education,Read intervention/solution text; do not infer ...
1,I2,Healthcare-worker training and clinical improv...,Read intervention/solution text; do not infer ...
2,I3,Referral and diagnostic pathway improvement,Read intervention/solution text; do not infer ...
3,I4,"Diagnostic facilities, equipment and workforce",Read intervention/solution text; do not infer ...
4,I5,"Financial, geographic and community access sup...",Read intervention/solution text; do not infer ...
5,I6,"Health-system coordination, digital tools and ...",Read intervention/solution text; do not infer ...


,analysis_study_id,analysis_country,source_row_excel,Intervention / solution,Intervention level (clean),Evidence of effectiveness,Potential applicability to Nigeria,Adaptation required,Key lesson
0,NG01,Nigeria,2,Targeted investment in diagnostic infrastructu...,Health system/service organization,The study identifies geographic gaps but does ...,Potentially applicable in Nigeria: health-work...,Prioritise underserved locations; budget for e...,Improving diagnostic access requires functioni...
1,NG02,Nigeria,3,Improve public knowledge and awareness of GI c...,Patient/community,Not directly tested,Potentially applicable in Nigeria: community a...,Co-design culturally appropriate messages in l...,Cancer symptom education should be paired with...
2,NG02,Nigeria,4,Strengthen systems for prompt diagnosis and re...,Health system/service organization,Not directly tested,Potentially applicable in Nigeria: referral co...,Map local primary-to-specialist referral route...,"Clear referral criteria, timely escalation and..."
3,NG02,Nigeria,5,"Consider early detection programmes, including...",Policy/national governance,Not directly tested,Potentially applicable in Nigeria: targeted ea...,Define cancer-specific eligibility and evidenc...,Early-detection approaches need cancer-specifi...
4,NG03,Nigeria,6,Strategies to increase breast cancer knowledge,Patient/community,High breast cancer knowledge was significantly...,Potentially applicable in Nigeria: community a...,Co-design culturally appropriate messages in l...,Cancer symptom education should be paired with...
5,NG03,Nigeria,7,Promote breast awareness and appropriate breas...,Patient/community,BSE practice was associated with earlier detec...,Potentially applicable in Nigeria: community a...,Co-design culturally appropriate messages in l...,Cancer symptom education should be paired with...
6,NG03,Nigeria,8,Address socioeconomic barriers to timely diagn...,Health system/service organization + Policy/na...,Not directly tested,Requires contextual assessment,Identify sustainable funding and patient eligi...,Reducing patient costs may improve access to t...
7,NG04,Nigeria,9,Community interventions aimed at promoting ear...,Patient/community,During the period of more intensive community ...,Requires contextual assessment,Further contextual assessment needed: specify ...,Community engagement should address local beli...


## 7. Analysis workbook export

An Excel analysis workbook was generated containing the input-file manifest, pathway-reporting audit, Nigerian regional evidence, the 68 approved thematic decisions and rationales, final coded extraction tables, and thematic frequency summaries. The source extraction workbooks were not overwritten.


In [ ]:
OUT_CONTINUATION='Nigeria_UK_RQ1_Pathways_Approved_Thematic_Coding.xlsx'
with pd.ExcelWriter(OUT_CONTINUATION,engine='openpyxl') as writer:
    manifest.to_excel(writer,sheet_name='Authoritative inputs',index=False)
    coverage_audited.to_excel(writer,sheet_name='Audited stage coverage',index=False)
    regional_counts.to_excel(writer,sheet_name='Region presence',index=False)
    region_stage_matrix.to_excel(writer,sheet_name='Region stage coverage',index=False)
    rq1_narratives.to_excel(writer,sheet_name='RQ1 narrative audit',index=False)
    decision_register.to_excel(writer,sheet_name='68 approved decisions',index=False)
    s3.to_excel(writer,sheet_name='Sheet3 final coded',index=False)
    s4.to_excel(writer,sheet_name='Sheet4 final coded',index=False)
    final_theme_counts.to_excel(writer,sheet_name='Approved theme counts',index=False)
print('Created one final analysis workbook:',OUT_CONTINUATION)
print('Stored in the Colab working directory. Download only when you choose to; see optional cell below.')


Created one final analysis workbook: Nigeria_UK_RQ1_Pathways_Approved_Thematic_Coding.xlsx
Stored in the Colab working directory. Download only when you choose to; see optional cell below.


In [ ]:
# OPTIONAL — run only if you want the final Excel output saved to your own device.
# This cell is intentionally separate so it does not fill local storage automatically.
from google.colab import files
files.download(OUT_CONTINUATION)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Interpretation boundaries and research-question analyses

The analysis maintained the following distinctions:

- **RQ1 — Diagnostic pathways:** Study-level reporting frequencies were not treated as patient-level prevalence, delay rates or evidence that unreported pathway stages did not occur.
- **RQ2 — Factors and barriers:** Themes described the subject of extracted findings. Associations, adverse effects and causal relationships were not treated as interchangeable. A finding could receive more than one thematic code.
- **RQ3 — Nigeria–UK comparison:** Descriptive differences were interpreted in light of unequal publication counts, cancer types, care settings and reporting specificity rather than as pooled comparisons of patient outcomes.
- **RQ4 — Intervention evidence:** Interventions proposed or described in publications were distinguished from interventions formally evaluated for effectiveness.

The notebook retained the original automated `coding_status` for auditing and used the approved `final_coding_status` and `final_theme_codes` for thematic summaries. Later code cells generated additional RQ1 pathway summaries, RQ2 factor and regional tables, RQ3 cross-country reporting comparisons, and an RQ4 evidence-screening register. The RQ4 screen identified 91 entries for further interpretation; the screening labels were not themselves a validated effectiveness classification.


In [ ]:

# RQ1: Diagnostic pathways across Nigerian regions
# Uses the existing s1 and s2 DataFrames.

import pandas as pd

ng1 = s1.loc[s1["analysis_country"].eq("Nigeria")].copy()
ng2 = s2.loc[s2["analysis_country"].eq("Nigeria")].copy()

assert len(ng1) == len(ng2) == 42
assert ng1["analysis_study_id"].is_unique
assert ng2["analysis_study_id"].is_unique
assert set(ng1["analysis_study_id"]) == set(ng2["analysis_study_id"])

pathway_fields = [
    "Symptom recognition",
    "Appraisal",
    "Initial action",
    "First healthcare contact",
    "Referral initiated",
    "Diagnostic investigation",
    "Diagnosis confirmation",
    "Pathway to care",
    "Route to diagnosis"
]

assert set(pathway_fields).issubset(ng2.columns)

# Combine regional information and pathway descriptions.
rq1_narratives = ng1[
    ["analysis_study_id", "nigeria_region_clean",
     "Author", "Year"]
].merge(
    ng2[["analysis_study_id"] + pathway_fields],
    on="analysis_study_id",
    validate="one_to_one"
).sort_values("analysis_study_id").reset_index(drop=True)

assert len(rq1_narratives) == 42

# Illustrative patterns previously identified from
# the extracted study narratives.
# Patterns can overlap; these are not patient frequencies.

rq1_examples = {
    "Informal/alternative care before or alongside formal care":
        ["NG02","NG08","NG09","NG10","NG12",
         "NG14","NG30","NG40","NG42"],

    "Multiple contacts and onward referral":
        ["NG02","NG03","NG08","NG09","NG12",
         "NG26","NG33","NG34","NG38"],

    "Clinical/hospital presentation with diagnostic work-up":
        ["NG05","NG11","NG20","NG24",
         "NG28","NG29","NG32"],

    "Case-finding or health-check entry":
        ["NG16"],

    "Biopsy/histology as a diagnostic step":
        ["NG05","NG08","NG09","NG11","NG28","NG33"]
}

# Create a traceable pattern evidence register.
rq1_pattern_rows = [
    {"pattern": pattern, "analysis_study_id": sid}
    for pattern, ids in rq1_examples.items()
    for sid in ids
]

rq1_patterns = pd.DataFrame(rq1_pattern_rows).merge(
    rq1_narratives,
    on="analysis_study_id",
    how="left",
    validate="many_to_one"
)

assert rq1_patterns["nigeria_region_clean"].notna().all()

rq1_pattern_summary = (
    rq1_patterns.groupby("pattern", sort=False)
    .agg(
        example_studies=("analysis_study_id", "nunique"),
        supporting_study_ids=(
            "analysis_study_id",
            lambda x: ", ".join(sorted(set(x)))
        )
    )
    .reset_index()
)

print("RQ1: Nigerian studies verified:",
      rq1_narratives["analysis_study_id"].nunique())

print("\nFive illustrative pathway patterns:")
display(rq1_pattern_summary)

print("\nSupporting evidence for each pattern:")
display(
    rq1_patterns[
        ["pattern", "analysis_study_id",
         "nigeria_region_clean",
         "Pathway to care", "Route to diagnosis"]
    ]
)

print("\nComplete 42-study pathway register:")
display(rq1_narratives)

print("RQ1 ANALYSIS COMPLETED")


RQ1: Nigerian studies verified: 42

Five illustrative pathway patterns:


,pattern,example_studies,supporting_study_ids
0,Informal/alternative care before or alongside ...,9,"NG02, NG08, NG09, NG10, NG12, NG14, NG30, NG40..."
1,Multiple contacts and onward referral,9,"NG02, NG03, NG08, NG09, NG12, NG26, NG33, NG34..."
2,Clinical/hospital presentation with diagnostic...,7,"NG05, NG11, NG20, NG24, NG28, NG29, NG32"
3,Case-finding or health-check entry,1,NG16
4,Biopsy/histology as a diagnostic step,6,"NG05, NG08, NG09, NG11, NG28, NG33"



Supporting evidence for each pattern:


,pattern,analysis_study_id,nigeria_region_clean,Pathway to care,Route to diagnosis
0,Informal/alternative care before or alongside ...,NG02,south_west,Symptom onset → initial action (often hospital...,Tertiary-hospital diagnostic pathway; many pat...
1,Informal/alternative care before or alongside ...,NG08,south_east,Symptom onset → parent/caregiver decision and ...,"Routes included primary health care, private/g..."
2,Informal/alternative care before or alongside ...,NG09,north_west;south_west,First symptom → appraisal/decision → possible ...,Patients moved through a median of 3 health fa...
3,Informal/alternative care before or alongside ...,NG10,south_west,Symptom onset → decision to seek healthcare → ...,"Multiple routes including private hospitals, h..."
4,Informal/alternative care before or alongside ...,NG12,south_east,Symptom recognition → initial healthcare conta...,Initial modern care commonly began with a gene...
5,Informal/alternative care before or alongside ...,NG14,south_west,First breast symptom → appraisal → waiting/alt...,Predominantly self-detected breast symptoms fo...
6,Informal/alternative care before or alongside ...,NG30,south_west,Symptom recognition → appraisal/help-seeking →...,"Public/private hospital, traditional healer, c..."
7,Informal/alternative care before or alongside ...,NG40,south_south,Symptom onset → appraisal/help-seeking → possi...,Symptom → traditional practitioner/faith heale...
8,Informal/alternative care before or alongside ...,NG42,north_central,Symptom recognition → patient-mediated delay →...,Patient-mediated route included alternative re...
9,Multiple contacts and onward referral,NG02,south_west,Symptom onset → initial action (often hospital...,Tertiary-hospital diagnostic pathway; many pat...



Complete 42-study pathway register:


,analysis_study_id,nigeria_region_clean,Author,Year,Symptom recognition,Appraisal,Initial action,First healthcare contact,Referral initiated,Diagnostic investigation,Diagnosis confirmation,Pathway to care,Route to diagnosis
0,NG01,national_or_multiregional,Omisore et al.,2023,NR,NR,NR,NR,NR,"Mammography, breast ultrasound and ultrasound-...",Ultrasound-guided breast biopsy identified as ...,Access to diagnostic imaging services varies g...,Health-system access to diagnostic imaging and...
1,NG02,south_west,Onwuka et al.,2024,Patients identified/noticed their first sympto...,NR as a distinct stage,Most patients visited a hospital after noticin...,Hospital; 55% initially presented to a private...,Patients were referred from other facilities t...,NR in sufficient detail for pathway mapping,Clinically and/or histologically confirmed GI ...,Symptom onset → initial action (often hospital...,Tertiary-hospital diagnostic pathway; many pat...
2,NG03,north_central;south_west,Agodirin et al.,2021,Breast cancer was detected by patients; tumour...,Concerns regarding hospital treatment were ass...,Premorbid help-seeking preferences were assess...,Orthodox healthcare provider or alternative ca...,"Reasons for referral were collected, but refer...",NR as a detailed diagnostic sequence,Newly diagnosed breast cancer; detailed confir...,Detection → healthcare-seeking pathway involvi...,NaN
3,NG04,south_west,Olasehinde et al.,2023,NR,NR,NR,NR,NR,NR,NR,The study assessed trends in stage at diagnosi...,NaN
4,NG05,north_central,"Adeoye, Oladapo, et al.",2021,Cough 88.7%; weight loss 84.5%; shortness of b...,NR,NR,NR,NR,Chest radiography 100%; chest CT 56.3%; pleura...,Tissue diagnosis/histopathological confirmation,Symptoms/signs → healthcare presentation → rad...,Clinical presentation → chest radiography/CT →...
5,NG06,south_east,Okeke et al.,2024,NR,NR,NR,Hospital presentation/first visit,NR,NR,Histological confirmation/biopsy,Symptoms → hospital presentation → first visit...,Hospital-based diagnosis following presentatio...
6,NG07,south_west,Oyetunde,2010,Cancer perceived as a major health problem; pe...,Perceived causes included nutritional patterns...,"Choice of cancer care influenced by sex, age, ...",Orthodox healthcare and other available health...,NR,NR,NR,Community perception → choice of cancer care →...,Orthodox medical care was preferred/used by 60...
7,NG08,south_east,Bartholomew F. Chukwu; Osita U. Ezenwosu; Anth...,2014,Symptoms varied by cancer; systemic symptoms s...,Parental recognition/appraisal contributed sub...,Visits to health professionals and alternative...,"Various first facilities: mission hospital, pr...","Yes, for patients coming from non-cancer treat...",Diagnostic tests; histological/cytological ass...,Histological or cytological confirmation,Symptom onset → parent/caregiver decision and ...,"Routes included primary health care, private/g..."
8,NG09,north_west;south_west,Fayehun et al.,2025,First noticeable cancer symptom/body change wa...,Some participants perceived symptoms as minor/...,Some delayed formal care; others sought family...,Formal healthcare professional; participants c...,Failure to refer identified as an important so...,Diagnostic investigations culminating in histo...,Histological confirmation.,First symptom → appraisal/decision → possible ...,Patients moved through a median of 3 health fa...
9,NG10,south_west,Rabiu et al.,2013,"Symptoms included abdominal swelling (97.3%), ...",Authors suggest limited awareness of ovarian c...,Patients sought care at various facilities; 21...,"Private hospital (35.1%), herbal home (21.6%),...",Referral to LASUTH from initial healthcare fac...,Surgical evaluation/staging; authors note that...,Histological confirmation; surgical/histologic...,Symptom onset → decision to seek healthcare → ...,"Multiple routes including private hospitals, h..."


RQ1 ANALYSIS COMPLETED


In [ ]:

# RQ2: Factors associated with diagnostic delays in Nigeria
# Reuses approved thematic coding in s3.
# Does not repeat or modify the coding process.

import pandas as pd

assert "final_theme_codes" in s3.columns
assert "final_coding_status" in s3.columns
assert not s3["final_coding_status"].eq("REVIEW").any()

rq2_factors = s3.loc[
    s3["analysis_country"].eq("Nigeria")
].copy()

assert len(rq2_factors) == 456
assert rq2_factors["analysis_study_id"].nunique() == 42

# Preserve traceable original findings.
rq2_fields = [
    "analysis_study_id",
    "source_row_excel",
    "Specific factor original",
    "Evidence/Finding original",
    "effect_direction_clean",
    "effect_evidence_language",
    "effect_outcome_clean",
    "pathway_stage_primary",
    "final_theme_codes",
    "final_coding_status"
]

assert set(rq2_fields).issubset(rq2_factors.columns)

rq2_evidence = rq2_factors[rq2_fields].copy()

# Approved eight-theme framework.
rq2_theme_names = {
    "T1": "Patient awareness/help-seeking",
    "T2": "Social/financial factors",
    "T3": "Healthcare-provider factors",
    "T4": "Referral/diagnostic processes",
    "T5": "Health-system/diagnostic capacity",
    "T6": "Geographic accessibility",
    "T7": "Clinical/disease factors",
    "T8": "Communication/information"
}

# Separate multiple approved themes.
rq2_long = rq2_factors.copy()

rq2_long["theme_code"] = (
    rq2_long["final_theme_codes"]
    .fillna("")
    .str.split(";")
)

rq2_long = rq2_long.explode("theme_code")
rq2_long["theme_code"] = (
    rq2_long["theme_code"].astype(str).str.strip()
)

rq2_long = rq2_long.loc[
    rq2_long["theme_code"].isin(rq2_theme_names)
].copy()

rq2_long["theme_name"] = (
    rq2_long["theme_code"].map(rq2_theme_names)
)

rq2_long = rq2_long.drop_duplicates(
    ["source_row_excel", "theme_code"]
)

# Define separate evidence categories.
conditions = {
    "All coded findings":
        pd.Series(True, index=rq2_long.index),

    "Adverse/delaying direction":
        rq2_long["effect_direction_clean"].eq(
            "delaying_or_adverse"
        ),

    "Reported association AND adverse":
        (
            rq2_long["effect_direction_clean"].eq(
                "delaying_or_adverse"
            )
            &
            rq2_long["effect_evidence_language"].eq(
                "reported_association"
            )
        ),

    "Reported null association":
        rq2_long["effect_evidence_language"].eq(
            "reported_null_association"
        )
}

# Summarise by theme and evidence group.
rq2_summary_parts = []

for label, mask in conditions.items():
    temp = (
        rq2_long.loc[mask]
        .groupby(["theme_code", "theme_name"])
        .agg(
            finding_rows=("source_row_excel", "nunique"),
            unique_studies=("analysis_study_id", "nunique")
        )
        .reset_index()
    )

    temp.insert(0, "evidence_group", label)
    rq2_summary_parts.append(temp)

rq2_theme_summary = pd.concat(
    rq2_summary_parts, ignore_index=True
)

# Include themes with zero findings.
full_grid = pd.MultiIndex.from_product(
    [list(conditions), list(rq2_theme_names)],
    names=["evidence_group", "theme_code"]
).to_frame(index=False)

rq2_theme_summary = full_grid.merge(
    rq2_theme_summary.drop(columns="theme_name"),
    on=["evidence_group", "theme_code"],
    how="left"
).fillna({
    "finding_rows": 0,
    "unique_studies": 0
})

rq2_theme_summary["theme_name"] = (
    rq2_theme_summary["theme_code"].map(rq2_theme_names)
)

rq2_theme_summary[
    ["finding_rows", "unique_studies"]
] = rq2_theme_summary[
    ["finding_rows", "unique_studies"]
].astype(int)

# Summarise effect direction and evidence strength.
rq2_direction_evidence = (
    rq2_factors.groupby(
        ["effect_direction_clean", "effect_evidence_language"],
        dropna=False
    )
    .agg(
        finding_rows=("analysis_study_id", "size"),
        unique_studies=("analysis_study_id", "nunique")
    )
    .reset_index()
    .sort_values("finding_rows", ascending=False)
)

# Final verification checks.
adverse = rq2_factors["effect_direction_clean"].eq(
    "delaying_or_adverse"
)

association = rq2_factors["effect_evidence_language"].eq(
    "reported_association"
)

null_association = rq2_factors[
    "effect_evidence_language"
].eq("reported_null_association")

assert adverse.sum() == 286
assert (adverse & association).sum() == 52
assert null_association.sum() == 11

print("Nigerian factor-level findings:", len(rq2_factors))
print("Unique Nigerian studies:",
      rq2_factors["analysis_study_id"].nunique())
print("Adverse/delaying findings:", adverse.sum())
print("Adverse reported associations:",
      (adverse & association).sum())
print("Reported null associations:",
      null_association.sum())

print("\nRQ2: Approved theme frequencies")
display(rq2_theme_summary)

print("\nRQ2: Effect direction and evidence classification")
display(rq2_direction_evidence)

print("\nRQ2: Original evidence register")
display(rq2_evidence)

print("RQ2 ANALYSIS COMPLETED")


Nigerian factor-level findings: 456
Unique Nigerian studies: 42
Adverse/delaying findings: 286
Adverse reported associations: 52
Reported null associations: 11

RQ2: Approved theme frequencies


,evidence_group,theme_code,finding_rows,unique_studies,theme_name
0,All coded findings,T1,44,15,Patient awareness/help-seeking
1,All coded findings,T2,198,38,Social/financial factors
2,All coded findings,T3,37,17,Healthcare-provider factors
3,All coded findings,T4,2,2,Referral/diagnostic processes
4,All coded findings,T5,129,36,Health-system/diagnostic capacity
5,All coded findings,T6,22,16,Geographic accessibility
6,All coded findings,T7,24,17,Clinical/disease factors
7,All coded findings,T8,2,1,Communication/information
8,Adverse/delaying direction,T1,33,11,Patient awareness/help-seeking
9,Adverse/delaying direction,T2,117,31,Social/financial factors



RQ2: Effect direction and evidence classification


,effect_direction_clean,effect_evidence_language,finding_rows,unique_studies
0,delaying_or_adverse,effect_stated_basis_unclear,176,31
4,direction_not_clear,effect_stated_basis_unclear,84,21
1,delaying_or_adverse,possible_or_hypothesized,55,22
2,delaying_or_adverse,reported_association,52,16
10,facilitating_or_shortening,effect_stated_basis_unclear,31,9
5,direction_not_clear,possible_or_hypothesized,21,14
14,no_clear_association,reported_null_association,10,3
6,direction_not_clear,reported_association,9,6
11,facilitating_or_shortening,possible_or_hypothesized,7,4
12,facilitating_or_shortening,reported_association,4,3



RQ2: Original evidence register


,analysis_study_id,source_row_excel,Specific factor original,Evidence/Finding original,effect_direction_clean,effect_evidence_language,effect_outcome_clean,pathway_stage_primary,final_theme_codes,final_coding_status
0,NG01,2,Unequal geographic access to diagnostic imaging,Access to mammography and ultrasound-guided bi...,delaying_or_adverse,possible_or_hypothesized,diagnosis_or_diagnostic_interval,diagnostic_investigation,T6,MAPPED_BY_RULE
1,NG01,3,Limited availability of ultrasound-guided brea...,"Only 33 of 1,336 facilities providing breast u...",direction_not_clear,effect_stated_basis_unclear,other_or_unspecified,diagnosis_confirmation,T5,MAPPED_BY_RULE
2,NG01,4,"Poorer diagnostic access in northern Nigeria, ...",Population-level access to mammography and ult...,facilitating_or_shortening,effect_stated_basis_unclear,diagnosis_or_diagnostic_interval,diagnostic_investigation,T2,MAPPED_BY_RULE
3,NG02,5,Belief that symptoms represented another diagn...,"Among participants with available data, belief...",delaying_or_adverse,effect_stated_basis_unclear,presentation_or_help_seeking,initial_action_help_seeking,T7,MAPPED_BY_RULE
4,NG02,6,Management for other conditions,"Among participants with available data, manage...",delaying_or_adverse,effect_stated_basis_unclear,diagnosis_or_diagnostic_interval,diagnosis_confirmation,T3,MAPPED_BY_RULE
...,...,...,...,...,...,...,...,...,...,...
451,NG42,453,Missed appointments,93 (18.49%); authors note strikes/public holid...,delaying_or_adverse,effect_stated_basis_unclear,other_or_unspecified,not_determined,T5,MAPPED_BY_RULE
452,NG42,454,Comorbid factors,48 (9.54%) reported comorbid factors,delaying_or_adverse,effect_stated_basis_unclear,treatment,treatment_initiation,T7,MAPPED_BY_RULE
453,NG42,455,Failure to recognize/act appropriately at firs...,Authors identify inadequate recognition/action...,delaying_or_adverse,effect_stated_basis_unclear,other_or_unspecified,initial_action_help_seeking,T5,MAPPED_BY_RULE
454,NG42,456,Poor cancer symptom awareness,Authors conclude cancer symptom awareness was ...,delaying_or_adverse,effect_stated_basis_unclear,presentation_or_help_seeking,symptom_recognition,T1,MAPPED_BY_RULE


RQ2 ANALYSIS COMPLETED


In [ ]:

# RQ2: Regional comparison of diagnostic-delay factors
# Uses authoritative Sheet 1 and approved Sheet 3 coding.

import pandas as pd

ng_studies = s1.loc[
    s1["analysis_country"] == "Nigeria",
    ["analysis_study_id", "nigeria_region_clean"]
].copy()

ng_factors = s3.loc[
    s3["analysis_country"] == "Nigeria"
].copy()

assert len(ng_studies) == 42
assert ng_studies["analysis_study_id"].is_unique
assert len(ng_factors) == 456
assert not ng_factors["final_coding_status"].eq("REVIEW").any()

# Connect each finding to its study's region.
regional = ng_factors.merge(
    ng_studies,
    on="analysis_study_id",
    how="left",
    validate="many_to_one"
)

# Split explicitly named multiregional combinations.
regional["region"] = (
    regional["nigeria_region_clean"]
    .fillna("NR")
    .astype(str)
    .str.split(";")
)
regional = regional.explode("region")
regional["region"] = regional["region"].str.strip()

# Separate findings with multiple approved theme codes.
regional["theme_code"] = (
    regional["final_theme_codes"]
    .fillna("")
    .str.split(";")
)
regional = regional.explode("theme_code")
regional["theme_code"] = regional["theme_code"].str.strip()

valid_themes = {f"T{i}" for i in range(1, 9)}
regional = regional.loc[
    regional["theme_code"].isin(valid_themes)
].copy()

# Avoid duplicate theme assignments within a finding.
regional = regional.drop_duplicates(
    ["region", "source_row_excel", "theme_code"]
)

# Separate adverse direction and reported association.
regional["adverse"] = (
    regional["effect_direction_clean"] == "delaying_or_adverse"
)

regional["adverse_association"] = (
    regional["adverse"]
    &
    regional["effect_evidence_language"].eq(
        "reported_association"
    )
)

# Count distinct studies by region and theme.
all_themes = (
    regional.groupby(["region", "theme_code"])
    ["analysis_study_id"]
    .nunique()
    .reset_index(name="studies_reporting_theme")
)

adverse_themes = (
    regional.loc[regional["adverse"]]
    .groupby(["region", "theme_code"])
    ["analysis_study_id"]
    .nunique()
    .reset_index(name="studies_reporting_adverse")
)

association_themes = (
    regional.loc[regional["adverse_association"]]
    .groupby(["region", "theme_code"])
    ["analysis_study_id"]
    .nunique()
    .reset_index(name="studies_with_reported_association")
)

rq2_regional_summary = (
    all_themes
    .merge(adverse_themes,
           on=["region", "theme_code"], how="left")
    .merge(association_themes,
           on=["region", "theme_code"], how="left")
    .fillna(0)
)

for column in [
    "studies_reporting_adverse",
    "studies_with_reported_association"
]:
    rq2_regional_summary[column] = (
        rq2_regional_summary[column].astype(int)
    )

print("RQ2 REGIONAL SUMMARY")
display(rq2_regional_summary)

print("ADVERSE FACTORS BY REGION")
rq2_regional_pivot = (
    rq2_regional_summary.pivot(
        index="region",
        columns="theme_code",
        values="studies_reporting_adverse"
    ).fillna(0).astype(int)
)

display(rq2_regional_pivot)

print("REPORTED ADVERSE ASSOCIATIONS BY REGION")
display(
    rq2_regional_summary.pivot(
        index="region",
        columns="theme_code",
        values="studies_with_reported_association"
    ).fillna(0).astype(int)
)

print("RQ2 REGIONAL ANALYSIS COMPLETED")


RQ2 REGIONAL SUMMARY


,region,theme_code,studies_reporting_theme,studies_reporting_adverse,studies_with_reported_association
0,NR,T2,1,0,0
1,NR,T5,1,0,0
2,national_or_multiregional,T1,1,0,0
3,national_or_multiregional,T2,2,1,1
4,national_or_multiregional,T5,1,0,0
5,national_or_multiregional,T6,2,1,0
6,national_or_multiregional,T7,1,0,0
7,north_central,T1,3,2,1
8,north_central,T2,4,3,3
9,north_central,T5,3,3,1


ADVERSE FACTORS BY REGION


theme_code,T1,T2,T3,T4,T5,T6,T7,T8
region,,,,,,,,
NR,0,0,0,0,0,0,0,0
national_or_multiregional,0,1,0,0,0,1,0,0
north_central,2,3,0,0,3,0,1,0
north_west,2,5,3,0,6,2,1,0
south_east,1,4,3,0,4,1,0,0
south_south,2,2,1,0,4,1,1,0
south_west,5,19,10,0,16,7,7,0


REPORTED ADVERSE ASSOCIATIONS BY REGION


theme_code,T1,T2,T3,T4,T5,T6,T7,T8
region,,,,,,,,
NR,0,0,0,0,0,0,0,0
national_or_multiregional,0,1,0,0,0,0,0,0
north_central,1,3,0,0,1,0,0,0
north_west,0,0,0,0,1,0,0,0
south_east,1,1,1,0,1,0,0,0
south_south,0,0,0,0,0,0,0,0
south_west,4,6,1,0,3,3,3,0


RQ2 REGIONAL ANALYSIS COMPLETED


In [ ]:

# RQ3 — Nigeria–UK diagnostic pathway comparison
# No new cleaning or thematic coding.

import pandas as pd

assert len(s2) == 242
assert s2["analysis_study_id"].nunique() == 242

# 1. Compare reporting for identical pathway stages.
rows = []

for stage, column in AUDIT_COLS.items():
    for country in ["Nigeria", "UK"]:
        subset = s2[s2["analysis_country"] == country]
        statuses = subset[column].map(
            lambda x: clear_status(column, x)
        )
        counts = statuses.value_counts()
        n = len(subset)

        rows.append({
            "country": country,
            "stage": stage,
            "total_studies": n,
            "explicit": counts.get("explicitly described", 0),
            "broad_unclear": counts.get("broad / unclear", 0),
            "not_reported": counts.get("not reported / NA", 0),
            "explicit_percent": round(
                100 * counts.get("explicitly described", 0) / n, 1
            )
        })

rq3_coverage = pd.DataFrame(rows)

assert (
    rq3_coverage[
        ["explicit", "broad_unclear", "not_reported"]
    ].sum(axis=1) == rq3_coverage["total_studies"]
).all()

print("RQ3: Pathway reporting comparison")
display(rq3_coverage)

# 2. Compare exact route and referral categories.
comparison_fields = {
    "First contact": "first_healthcare_contact_clean",
    "Referral": "referral_initiated_clean",
    "Referral source": "referral_source_clean",
    "Referral destination": "referral_destination_clean",
    "Route to diagnosis": "route_to_diagnosis_clean"
}

results = []

for label, column in comparison_fields.items():
    for country in ["Nigeria", "UK"]:
        subset = s2[s2["analysis_country"] == country]
        counts = subset[column].fillna("Missing").value_counts()

        for category, count in counts.items():
            results.append({
                "component": label,
                "country": country,
                "category": category,
                "unique_studies": count,
                "percentage": round(
                    100 * count / len(subset), 1
                )
            })

rq3_routes = pd.DataFrame(results)

print("RQ3: Referral and route categories")
display(rq3_routes)

# 3. Keep original study IDs for traceability.
rq3_evidence = s2[[
    "analysis_study_id",
    "analysis_country",
    "First healthcare contact",
    "Referral initiated",
    "Referral source",
    "Referral destination",
    "Route to diagnosis"
]].copy()

print("RQ3: Study-level supporting evidence")
display(rq3_evidence)

# 4. Save results without automatically downloading.
rq3_coverage.to_csv("RQ3_pathway_comparison.csv", index=False)
rq3_routes.to_csv("RQ3_route_comparison.csv", index=False)
rq3_evidence.to_csv("RQ3_study_evidence.csv", index=False)

print("RQ3 ANALYSIS COMPLETED")


RQ3: Pathway reporting comparison


,country,stage,total_studies,explicit,broad_unclear,not_reported,explicit_percent
0,Nigeria,Symptom recognition,42,31,1,10,73.8
1,UK,Symptom recognition,200,129,30,41,64.5
2,Nigeria,Appraisal,42,10,10,22,23.8
3,UK,Appraisal,200,2,169,29,1.0
4,Nigeria,Initial action,42,20,5,17,47.6
5,UK,Initial action,200,132,37,31,66.0
6,Nigeria,First healthcare contact,42,26,7,9,61.9
7,UK,First healthcare contact,200,179,15,6,89.5
8,Nigeria,Referral initiated,42,11,5,26,26.2
9,UK,Referral initiated,200,137,40,23,68.5


RQ3: Referral and route categories


,component,country,category,unique_studies,percentage
0,First contact,Nigeria,hospital_or_formal_care_level_unspecified,13,31.0
1,First contact,Nigeria,not_reported,9,21.4
2,First contact,Nigeria,specialist_or_tertiary_care,6,14.3
3,First contact,Nigeria,other_or_unclear,6,14.3
4,First contact,Nigeria,multiple_or_variable_first_contacts,4,9.5
5,First contact,Nigeria,alternative_or_traditional_care,2,4.8
6,First contact,Nigeria,not_determined_from_description,1,2.4
7,First contact,Nigeria,emergency_department,1,2.4
8,First contact,UK,primary_care_or_gp,135,67.5
9,First contact,UK,multiple_or_variable_first_contacts,20,10.0


RQ3: Study-level supporting evidence


,analysis_study_id,analysis_country,First healthcare contact,Referral initiated,Referral source,Referral destination,Route to diagnosis
0,NG01,Nigeria,NR,NR,NR,Diagnostic imaging facility,Health-system access to diagnostic imaging and...
1,NG02,Nigeria,Hospital; 55% initially presented to a private...,Patients were referred from other facilities t...,Mainly private hospitals for 55% of patients,Obafemi Awolowo University Teaching Hospitals ...,Tertiary-hospital diagnostic pathway; many pat...
2,NG03,Nigeria,Orthodox healthcare provider or alternative ca...,"Reasons for referral were collected, but refer...",NR,Referral centres,NaN
3,NG04,Nigeria,NR,NR,NR,NR,NaN
4,NG05,Nigeria,NR,NR,NR,University of Ilorin Teaching Hospital/cardiot...,Clinical presentation → chest radiography/CT →...
...,...,...,...,...,...,...,...
237,UK196,UK,Primary care and other healthcare services,Yes,GP/primary care and other routes,Paediatric/secondary/tertiary cancer services,Multiple routes to diagnosis
238,UK197,UK,Primary care,Referral was part of included pathway evidence,Patient → GP,Secondary care,Multiple routes/pathways across reviewed studies
239,UK198,UK,Community/primary care-linked,Planned,Planned participant-reported route,Planned healthcare pathway,Not yet evaluated
240,UK199,UK,Primary care/dental care,Referral patterns reviewed,GP and dentist,Specialist care,GP and dental referral routes


RQ3 ANALYSIS COMPLETED


In [ ]:

# RQ4: Screening intervention evidence in Sheet 4
# No new extraction, cleaning, or coding required.

import pandas as pd

rq4 = s4.copy()

evidence_col = "Evidence of effectiveness "
intervention_col = "Intervention / solution "
applicability_col = "Potential applicability to Nigeria"

required = [
    "analysis_study_id", "analysis_country",
    "source_row_excel", evidence_col,
    intervention_col, applicability_col,
    "Problem category (clean)"
]

assert all(c in rq4.columns for c in required)
assert len(rq4) == 472
assert rq4["analysis_study_id"].nunique() == 242

def classify_evidence(value):
    if pd.isna(value) or str(value).strip().upper() in {
        "NR", "N/R", "NOT REPORTED", ""
    }:
        return "Not reported"

    t = str(value).strip().lower()

    not_evaluated = [
        "not directly tested",
        "not formally tested",
        "not evaluated",
        "no intervention evaluated",
        "not an intervention evaluation",
        "not an intervention study",
        "not intervention evaluation",
        "no effectiveness trial",
        "not an intervention-effectiveness trial",
        "recommendation only",
        "recommendation rather than",
        "recommendation/inference",
        "not a trial",
        "modelling evidence",
        "modeling evidence",
        "not experimentally evaluated",
        "not an evaluated intervention"
    ]

    if any(term in t for term in not_evaluated):
        return "Not directly evaluated / recommendation"

    if (
        "evaluated intervention" in t
        or t.startswith("evaluated;")
        or "intervention evaluation" in t
        or "trial implementation" in t
        or "experimental evidence" in t
    ):
        return "Evaluation mentioned — inspect outcome"

    return "Other evidence — inspect manually"

rq4["evidence_screen"] = rq4[evidence_col].apply(
    classify_evidence
)

# Count entries and distinct studies separately.
rq4_summary = (
    rq4.groupby(
        ["analysis_country", "evidence_screen"]
    )
    .agg(
        finding_rows=("analysis_study_id", "size"),
        unique_studies=("analysis_study_id", "nunique")
    )
    .reset_index()
)

print("RQ4: Intervention evidence screening")
display(rq4_summary)

# Identify illustrative practices for interpretation.
example_ids = [
    "NG04", "NG03", "UK006", "UK010",
    "UK016", "UK018", "UK048",
    "UK068", "UK070", "UK156"
]

rq4_examples = rq4.loc[
    rq4["analysis_study_id"].isin(example_ids),
    [
        "analysis_study_id",
        "source_row_excel",
        intervention_col,
        evidence_col,
        "evidence_screen",
        applicability_col
    ]
].sort_values(
    ["analysis_study_id", "source_row_excel"]
)

print("RQ4: Intervention examples with original evidence")
display(rq4_examples)

# Preserve complete original records.
rq4_register = rq4.copy()

# Save results without automatically downloading.
rq4_summary.to_csv(
    "RQ4_evidence_summary.csv", index=False
)
rq4_examples.to_csv(
    "RQ4_intervention_examples.csv", index=False
)
rq4_register.to_csv(
    "RQ4_complete_evidence_register.csv", index=False
)

print("RQ4 SCREENING COMPLETED")


RQ4: Intervention evidence screening


,analysis_country,evidence_screen,finding_rows,unique_studies
0,Nigeria,Not directly evaluated / recommendation,237,39
1,Nigeria,Not reported,6,2
2,Nigeria,Other evidence — inspect manually,5,4
3,UK,Evaluation mentioned — inspect outcome,10,10
4,UK,Not directly evaluated / recommendation,44,42
5,UK,Not reported,94,76
6,UK,Other evidence — inspect manually,76,73


RQ4: Intervention examples with original evidence


,analysis_study_id,source_row_excel,Intervention / solution,Evidence of effectiveness,evidence_screen,Potential applicability to Nigeria
4,NG03,6,Strategies to increase breast cancer knowledge,High breast cancer knowledge was significantly...,Other evidence — inspect manually,Potentially applicable in Nigeria: community a...
5,NG03,7,Promote breast awareness and appropriate breas...,BSE practice was associated with earlier detec...,Other evidence — inspect manually,Potentially applicable in Nigeria: community a...
6,NG03,8,Address socioeconomic barriers to timely diagn...,Not directly tested,Not directly evaluated / recommendation,Requires contextual assessment
7,NG04,9,Community interventions aimed at promoting ear...,During the period of more intensive community ...,Other evidence — inspect manually,Requires contextual assessment
264,UK006,266,Fast-track/Two-Week Wait referral pathway for ...,Demonstrated shorter time to specialist review...,Other evidence — inspect manually,Potentially applicable in Nigeria: referral co...
265,UK006,267,Strategies to reduce emergency presentation an...,NR,Not reported,Potentially applicable in Nigeria: referral co...
278,UK010,280,Establishment of a Rapid Diagnostic Clinic (RD...,Evaluated intervention. Referral-to-consultati...,Evaluation mentioned — inspect outcome,Potentially applicable in Nigeria: diagnostic ...
285,UK016,287,Targeted GP continuing medical education,Evaluated; ineffective in this study. Correct ...,Evaluation mentioned — inspect outcome,Potentially applicable in Nigeria: community a...
287,UK018,289,Standardised e-referral form + NHS e-RS + Trus...,Evaluated within NHS-Galleri trial implementat...,Evaluation mentioned — inspect outcome,Potentially applicable in Nigeria: referral co...
317,UK048,319,Straight-to-test colorectal pathway,Evaluated intervention: median diagnosis time ...,Evaluation mentioned — inspect outcome,Potentially applicable in Nigeria: referral co...


RQ4 SCREENING COMPLETED


In [ ]:

# Display effectiveness evidence requiring interpretation

review = rq4.loc[
    rq4["evidence_screen"].isin([
        "Evaluation mentioned — inspect outcome",
        "Other evidence — inspect manually"
    ]),
    [
        "analysis_study_id",
        "analysis_country",
        "Intervention / solution ",
        "Evidence of effectiveness ",
        "evidence_screen"
    ]
].copy()

print("Entries requiring interpretation:", len(review))

display(review)


Entries requiring interpretation: 91


,analysis_study_id,analysis_country,Intervention / solution,Evidence of effectiveness,evidence_screen
0,NG01,Nigeria,Targeted investment in diagnostic infrastructu...,The study identifies geographic gaps but does ...,Other evidence — inspect manually
4,NG03,Nigeria,Strategies to increase breast cancer knowledge,High breast cancer knowledge was significantly...,Other evidence — inspect manually
5,NG03,Nigeria,Promote breast awareness and appropriate breas...,BSE practice was associated with earlier detec...,Other evidence — inspect manually
7,NG04,Nigeria,Community interventions aimed at promoting ear...,During the period of more intensive community ...,Other evidence — inspect manually
134,NG24,Nigeria,Expand similar research across Nigeria's geopo...,Not applicable — research recommendation,Other evidence — inspect manually
...,...,...,...,...,...
446,UK175,UK,"Improve symptom awareness, access and navigati...",Qualitative study; recommendations/implication...,Other evidence — inspect manually
460,UK189,UK,Cancer awareness campaigns and messages encour...,Not formally evaluated in this study; authors ...,Other evidence — inspect manually
464,UK193,UK,Improve referral selection/clinical effectiven...,Observational audit; no intervention tested,Other evidence — inspect manually
465,UK194,UK,GPs should actively elicit normalised symptoms...,Recommendation based on qualitative findings; ...,Other evidence — inspect manually


In [ ]:

# RQ4: Export the 91 entries requiring interpretation

rq4_review = rq4.loc[
    rq4["evidence_screen"].isin([
        "Evaluation mentioned — inspect outcome",
        "Other evidence — inspect manually"
    ])
].copy()

assert len(rq4_review) == 91

columns = [
    "analysis_study_id",
    "analysis_country",
    "source_row_excel",
    "Intervention / solution ",
    "Evidence of effectiveness ",
    "Potential applicability to Nigeria",
    "evidence_screen"
]

rq4_review = rq4_review[columns]

print("Entries for review:", len(rq4_review))
print("\nBy country:")
display(
    rq4_review.groupby("analysis_country")
    .agg(
        entries=("analysis_study_id", "size"),
        unique_studies=("analysis_study_id", "nunique")
    )
)

rq4_review.to_csv(
    "RQ4_91_Evidence_Review.csv",
    index=False,
    encoding="utf-8-sig"
)

from google.colab import files
files.download("RQ4_91_Evidence_Review.csv")

print("RQ4 REVIEW FILE CREATED")


Entries for review: 91

By country:


,entries,unique_studies
analysis_country,,
Nigeria,5,4
UK,86,83


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RQ4 REVIEW FILE CREATED
